# 企業の最高益はどこから来たのか：再現ノートブック

SVillage の記事「実質の売上はほとんど増えていないのに、企業はなぜ最高益なのか――法人企業統計で、経常利益124.8兆円の中身を調べる」の数値と図11枚を再現します。

**このノートブックにデータは含まれていません。** 公式の入手先から、実行時に取得します。

- 日本銀行（国際収支、ドル円、企業物価、サービス価格）、内閣府（国民経済計算）、財務省（純粋持株会社を除く経常利益）：自動で取得します。
- 法人企業統計（e-Stat）：e-Stat API の appId（無料）があれば自動で取得します。無い場合は、画面からCSVを3つダウンロードして、途中でアップロードします（下のステップ2）。

**使い方：** 上から順に、セルを実行します。数分かかります。最後に、記事の主な数値との照合結果が表示されます。

**データの出所：** [e-Stat 法人企業統計調査 時系列データ 表1](https://www.e-stat.go.jp/dbview?sid=0003060791)、[日本銀行 時系列統計データ検索サイト](https://www.stat-search.boj.or.jp/)、[内閣府 国民経済計算](https://www.esri.cao.go.jp/jp/sna/menu.html)、[財務総合政策研究所](https://www.mof.go.jp/pri/reference/ssc/results/data.htm)。各機関の利用規約・出典表示に従ってください。

**注意：** 統計は改定されます。記事執筆時（2026年10月10日）より後に取得すると、記事の数値と一致しないことがあります（とくに国際収支の再投資収益は、毎年11月に改訂されます）。

In [ ]:
# 準備：日本語フォントと依存パッケージ
import subprocess, sys, os
try:
    import japanize_matplotlib  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "japanize-matplotlib"], check=False)
    try:
        import japanize_matplotlib  # noqa: F401
    except ImportError:
        print("日本語フォントの導入に失敗しました。図の日本語が文字化けする場合は、ランタイムを再起動してください")
for d in ["data/raw", "data/estat", "charts"]:
    os.makedirs(d, exist_ok=True)
print("準備できました")

## 0. スクリプトを書き出す

下のセルは、計算スクリプト7本を書き出します。内容は、リポジトリのファイルと同じです。

In [ ]:
%%writefile fetch_data.py
"""公式の入手先から、日本銀行・内閣府・財務省のデータを data/raw/ に取得する。

このリポジトリは生データを含みません。実行すると、次を公式サイトから取得して保存します。
- 日本銀行「時系列統計データ検索サイト」API（認証不要）：国際収支、ドル円、国内企業物価指数、企業向けサービス価格指数
- 内閣府「国民経済計算」2024年度年次推計の Excel
- 財務省 財務総合政策研究所「純粋持株会社を除く経常利益」の Excel（検算用）

法人企業統計（e-Stat）は、画面からダウンロードしたCSVを使います（README参照）。

実行: python fetch_data.py
取得したデータの指紋を、記事執筆時（2026-10-10）の値と比べて表示します。
統計は改定されるため、不一致でも誤りとは限りません（数値が記事と変わる可能性があります）。
"""
import csv
import hashlib
import sys
import urllib.parse
import urllib.request
from pathlib import Path

HERE = Path(__file__).resolve().parent
RAW = HERE / "data" / "raw"
UA = {"User-Agent": "Mozilla/5.0 (compatible; reproduction-script)"}

BOJ_API = "https://www.stat-search.boj.or.jp/api/v1/getDataCode?"
BOJ_FILES = {
    "boj_bp_direct_investment_monthly.csv": dict(
        db="BP01", startDate="199601", endDate="202608",
        code="BPBP6JYNPIN,BPBP6JYNPIC,BPBP6JYNPIC2,BPBP6JYNPIC21,BPBP6JYNPID21,BPBP6JYNPIC211,"
             "BPBP6JYNPIC2111,BPBP6JYNPIC2112,BPBP6JYNPIC212,BPBP6JYNPIC22"),
    "boj_fx_usdjpy_monthly_avg.csv": dict(db="FM08", startDate="199601", endDate="202609", code="FXERM07"),
    "boj_cgpi_monthly.csv": dict(db="PR01", startDate="198001", endDate="202608", code="PRCG20_2200000000"),
    "boj_sppi_monthly.csv": dict(db="PR02", startDate="198501", endDate="202608", code="PRCS20_5200000000"),
}
URL_FILES = {
    "2024i2_jp.xlsx": "https://www.esri.cao.go.jp/jp/sna/data/data_list/kakuhou/files/2024/tables/2024i2_jp.xlsx",
    "keitunenenpo.xlsx": "https://www.mof.go.jp/pri/reference/ssc/results/keitunenenpo.xlsx",
}
# 記事執筆時（2026-10-10）に取得したデータの指紋（参考：統計の改定で変わることがある）
# Excelはファイルの SHA-256。日銀のCSVは、取得時刻などが毎回変わるため、
# 「系列コード・時点・値」だけから作った指紋で比べる。
REF_HASH = {
    "boj_bp_direct_investment_monthly.csv": "44b69115da8164e37201d293acadaede182e87fc02c22d6a38f7c1e46c67b31f",
    "boj_fx_usdjpy_monthly_avg.csv": "4e7311999091cf57a63524411f38609c64bf19c713285fc9afafbeb9cca8b237",
    "boj_cgpi_monthly.csv": "59a1e0925cfbf821224b097c619cb1ca26bbed2280dd8118e4b31cb3c89e06b8",
    "boj_sppi_monthly.csv": "734089b1c8859910f53258bfe883f4311dc64651b20c9335537d9f50f4e28e09",
    "2024i2_jp.xlsx": "81090bee1c247daa705072a5404a522298cc5e300976d497dd38ef3c075b3914",
    "keitunenenpo.xlsx": "97dff1553ef368d263529f6f07e8c96adee89fd3962424682b825c8c99e8c19e",
}


def fingerprint(name, data):
    if name.endswith(".csv"):
        lines = data.decode("cp932").splitlines()
        start = next(i for i, l in enumerate(lines) if l.startswith("SERIES_CODE,"))
        h = hashlib.sha256()
        for r in csv.reader(lines[start + 1:]):
            h.update(("|".join([r[0], r[6], r[7]]) + "\n").encode())
        return h.hexdigest()
    return hashlib.sha256(data).hexdigest()


def get(url):
    req = urllib.request.Request(url, headers=UA)
    with urllib.request.urlopen(req, timeout=120) as r:
        return r.read()


def main():
    RAW.mkdir(parents=True, exist_ok=True)
    jobs = []
    for name, params in BOJ_FILES.items():
        q = urllib.parse.urlencode({"format": "csv", "lang": "jp", **params}, safe=",")
        jobs.append((name, BOJ_API + q))
    for name, url in URL_FILES.items():
        jobs.append((name, url))
    ok = True
    for name, url in jobs:
        try:
            data = get(url)
        except Exception as e:  # noqa: BLE001
            print(f"NG  {name}: 取得に失敗しました（{e}）\n    {url}")
            ok = False
            continue
        (RAW / name).write_bytes(data)
        digest = fingerprint(name, data)
        note = "（記事執筆時と一致）" if digest == REF_HASH[name] else "（記事執筆時の値と異なる：統計の改定の可能性）"
        print(f"OK  {name}  {len(data):,} bytes  sha256={digest[:12]}…{note}")
    if not ok:
        sys.exit("一部のデータが取得できませんでした。")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile fetch_estat_api.py
"""e-Stat API から法人企業統計 表1を取得し、data/estat/ に3つのCSVを作る（parse_estat.py が読む形式）。

必要なもの：e-Stat の appId（無料。https://www.e-stat.go.jp/api/ で利用登録して取得）。
appId は、環境変数 ESTAT_APP_ID に入れるか、実行時の入力（画面に表示されません）で渡します。
appId はファイルに保存しません。他人に渡したり、リポジトリに入れたりしないでください。

取得するのは「法人企業統計調査 時系列データ 表1 金融業、保険業以外の業種（原数値）」（年度）です。
  statsDataId = 0003060791（e-Statの統計表ID。画面のURL dbview?sid=0003060791 と同じ）
業種と規模の組み合わせ3つ：
  estat_all_ex_fin_all_size.csv   全産業（除く金融保険業）・全規模
  estat_pure_holding_all_size.csv 純粋持株会社・全規模
  estat_all_ex_fin_10oku_plus.csv 全産業（除く金融保険業）・資本金10億円以上

実行: python fetch_estat_api.py
取得後、記事の値（2025年度・全産業・全規模の経常利益など）と照合して表示します。
統計が改定されていれば、不一致になることがあります。
"""
import csv
import getpass
import json
import os
import re
import sys
import urllib.error
import urllib.parse
import urllib.request
from pathlib import Path

HERE = Path(__file__).resolve().parent
ESTAT = HERE / "data" / "estat"

API = "https://api.e-stat.go.jp/rest/3.0/app/json/"
STATS_DATA_ID = "0003060791"
SELECTIONS = {
    "estat_all_ex_fin_all_size.csv": ("全産業（除く金融保険業）", "全規模"),
    "estat_pure_holding_all_size.csv": ("純粋持株会社", "全規模"),
    "estat_all_ex_fin_10oku_plus.csv": ("全産業（除く金融保険業）", "10億円以上"),
}
# 記事執筆時（2026-10-10）の値（百万円）。全産業（除く金融保険業）・全規模・2025年度
CHECK_2025 = {"経常利益": 124831342, "売上高": 1720255968, "付加価値": 373670064}


def norm(s):
    s = re.sub(r"[\s　]+", "", str(s))
    return s.replace("(", "（").replace(")", "）")


def call(app_id, method, **params):
    q = urllib.parse.urlencode({"appId": app_id, "lang": "J", **params})
    url = f"{API}{method}?{q}"
    try:
        with urllib.request.urlopen(url, timeout=120) as r:
            return json.loads(r.read().decode("utf-8"))
    except urllib.error.HTTPError as e:
        sys.exit(f"e-Stat API のエラー（HTTP {e.code}）。appId が正しいか確認してください。")
    except urllib.error.URLError as e:
        sys.exit(f"e-Stat API に接続できません（{e.reason}）。")


def as_list(x):
    return x if isinstance(x, list) else [x]


def parse_meta(meta):
    """軸（調査項目・業種・規模）と時間軸を、名前から探す。"""
    objs = as_list(meta["GET_META_INFO"]["METADATA_INF"]["CLASS_INF"]["CLASS_OBJ"])
    axes = {}
    for o in objs:
        name = norm(o["@name"])
        classes = {c["@code"]: c["@name"] for c in as_list(o["CLASS"])}
        if o["@id"] == "time":
            axes["time"] = (o["@id"], classes)
        elif name.startswith("業種"):
            axes["industry"] = (o["@id"], classes)
        elif name.startswith("規模"):
            axes["size"] = (o["@id"], classes)
        elif name.startswith("調査項目") or name.startswith("項目"):
            axes["item"] = (o["@id"], classes)
    missing = [k for k in ("time", "industry", "size", "item") if k not in axes]
    if missing:
        sys.exit(f"統計表の軸を特定できませんでした: {missing}。e-Statの統計表の構成が変わった可能性があります。")
    return axes


def code_of(classes, wanted):
    hits = [c for c, n in classes.items() if norm(n) == norm(wanted)]
    if not hits:
        sys.exit(f"「{wanted}」が見つかりません。候補: {[n for n in classes.values()][:15]}…")
    return hits[0]


def fetch_one(app_id, axes, industry, size):
    ind_id, ind_cls = axes["industry"]
    size_id, size_cls = axes["size"]
    params = {
        "statsDataId": STATS_DATA_ID,
        "cd" + ind_id[0].upper() + ind_id[1:]: code_of(ind_cls, industry),
        "cd" + size_id[0].upper() + size_id[1:]: code_of(size_cls, size),
        "metaGetFlg": "N", "cntGetFlg": "N", "limit": "100000",
    }
    rows = []
    start = None
    while True:
        p = dict(params)
        if start:
            p["startPosition"] = start
        d = call(app_id, "getStatsData", **p)
        sd = d["GET_STATS_DATA"]
        if sd["RESULT"].get("STATUS") not in (0, "0"):
            sys.exit(f"e-Stat API: {sd['RESULT'].get('ERROR_MSG')}")
        rows += as_list(sd["STATISTICAL_DATA"]["DATA_INF"]["VALUE"])
        nxt = sd["STATISTICAL_DATA"]["RESULT_INF"].get("NEXT_KEY")
        if not nxt:
            break
        start = nxt
    return rows


def write_csv(path, rows, axes):
    item_id, item_cls = axes["item"]
    time_cls = axes["time"][1]
    table = {}
    for v in rows:
        year = re.match(r"\d{4}", time_cls[v["@time"]])
        if not year:
            continue
        table.setdefault(int(year.group()), {})[v["@" + item_id]] = v.get("$", "")
    item_codes = list(item_cls)
    with open(path, "w", encoding="utf-8-sig", newline="") as f:
        w = csv.writer(f)
        w.writerow([""] + [item_cls[c] for c in item_codes])
        for y in sorted(table):
            w.writerow([f"{y}年度"] + [table[y].get(c, "") for c in item_codes])
    return table, item_cls


def main():
    app_id = os.environ.get("ESTAT_APP_ID") or getpass.getpass("e-Stat の appId（入力は表示されません）: ").strip()
    if not app_id:
        sys.exit("appId がありません。")
    ESTAT.mkdir(parents=True, exist_ok=True)
    meta = call(app_id, "getMetaInfo", statsDataId=STATS_DATA_ID)
    axes = parse_meta(meta)
    for fname, (industry, size) in SELECTIONS.items():
        rows = fetch_one(app_id, axes, industry, size)
        table, item_cls = write_csv(ESTAT / fname, rows, axes)
        print(f"OK  {fname}: {len(table)}年度、{len(item_cls)}項目（{industry}・{size}）")
        if fname == "estat_all_ex_fin_all_size.csv":
            names = {norm(re.sub(r"【[^】]*】|[（(]当期末[）)]", "", n)): c for c, n in item_cls.items()}
            for item, expected in CHECK_2025.items():
                code = names.get(item)
                got = table.get(2025, {}).get(code, "")
                ok = str(got).replace(",", "") == str(expected)
                print(f"  {'OK ' if ok else 'NG '} 2025年度の{item}: {got}（記事執筆時 {expected:,}）")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile parse_estat.py
"""e-Statの法人企業統計（時系列表1）からダウンロードしたファイル3つを読み、分析用の5つのCSVを data/raw/ に作る。

入力（data/estat/ に置く。ファイル名は自由。CSVでもExcel（.xlsx）でもよい）:
  業種＝全産業（除く金融保険業）、規模＝全規模
  業種＝純粋持株会社、規模＝全規模
  業種＝全産業（除く金融保険業）、規模＝資本金10億円以上
どの表かは、ファイルの中の「業種」「規模」の値から自動で判定します。

出力（data/raw/）:
  houjin_table1_all_ex_fin_FY1980_2025.csv
  houjin_table1_pure_holding_FY2009_2025.csv
  houjin_table1_all_ex_fin_value_added_FY1980_2025.csv
  houjin_table1_capital_10oku_plus_value_added_FY1980_2025.csv
  houjin_table1_all_ex_fin_cost_structure_FY1980_2025.csv

読める形：e-Statの「ダウンロード」の次の形式
  (1) CSV（項目・業種・規模・年度・値が縦に並ぶ形）
  (2) Excel（上に表の説明があり、行が年度・列が項目の形）
  のほか、行が年度・列が項目のCSV、行が項目・列が年度のCSVも読めます。
列名の「(当期末)」や「【百万円】」などの違いは吸収します。必要な項目が見つからなければ、エラーで止まります。

実行: python parse_estat.py
"""
import csv
import io
import re
import sys
import warnings
from pathlib import Path

import pandas as pd

HERE = Path(__file__).resolve().parent
ESTAT = HERE / "data" / "estat"
RAW = HERE / "data" / "raw"

# 判定できない形式（行が年度の表など）のときだけ使う、決まったファイル名
INPUTS = {
    "all": "estat_all_ex_fin_all_size.csv",
    "holding": "estat_pure_holding_all_size.csv",
    "large": "estat_all_ex_fin_10oku_plus.csv",
}
DESC = {
    "all": "業種＝全産業（除く金融保険業）、規模＝全規模",
    "holding": "業種＝純粋持株会社、規模＝全規模",
    "large": "業種＝全産業（除く金融保険業）、規模＝10億円以上",
}
SELECT = {
    ("全産業（除く金融保険業）", "全規模"): "all",
    ("純粋持株会社", "全規模"): "holding",
    ("全産業（除く金融保険業）", "10億円以上"): "large",
}

# 出力の列名 ← e-Statの項目名（正規化後）
ITEM = {
    "母集団_社": "母集団",
    "資産合計": "資産合計",
    "売上高": "売上高",
    "売上原価": "売上原価",
    "販売費及び一般管理費": "販売費及び一般管理費",
    "営業利益": "営業利益",
    "営業外収益": "営業外収益",
    "営業外費用": "営業外費用",
    "経常利益": "経常利益",
    "特別利益": "特別利益",
    "特別損失": "特別損失",
    "当期純利益": "当期純利益",
    "配当金計": "配当金計",
    "社内留保": "社内留保",
    "減価償却費計": "減価償却費計",
    "役員給与": "役員給与",
    "役員賞与": "役員賞与",
    "従業員給与": "従業員給与",
    "従業員賞与": "従業員賞与",
    "福利厚生費": "福利厚生費",
    "支払利息等": "支払利息等",
    "動産不動産賃借料": "動産・不動産賃借料",
    "租税公課": "租税公課",
    "期中平均役員数_人": "期中平均役員数",
    "期中平均従業員数_人": "期中平均従業員数",
    "付加価値": "付加価値",
    "営業純益": "営業純益",
}

OUTPUTS = {
    "houjin_table1_all_ex_fin_FY1980_2025.csv": ("all", 1980, [
        "母集団_社", "資産合計", "売上高", "営業利益", "営業外収益", "営業外費用", "経常利益", "特別利益", "特別損失",
        "当期純利益", "配当金計", "社内留保", "減価償却費計", "従業員給与", "従業員賞与", "福利厚生費", "支払利息等",
        "期中平均従業員数_人", "付加価値"]),
    "houjin_table1_pure_holding_FY2009_2025.csv": ("holding", 2009, [
        "母集団_社", "資産合計", "売上高", "営業利益", "営業外収益", "営業外費用", "経常利益", "配当金計", "支払利息等",
        "期中平均従業員数_人"]),
    "houjin_table1_all_ex_fin_value_added_FY1980_2025.csv": ("all", 1980, [
        "売上高", "営業利益", "減価償却費計", "役員給与", "役員賞与", "従業員給与", "従業員賞与", "福利厚生費", "支払利息等",
        "動産不動産賃借料", "租税公課", "期中平均役員数_人", "期中平均従業員数_人", "付加価値", "営業純益"]),
    "houjin_table1_capital_10oku_plus_value_added_FY1980_2025.csv": ("large", 1980, [
        "母集団_社", "売上高", "営業利益", "減価償却費計", "役員給与", "役員賞与", "従業員給与", "従業員賞与", "福利厚生費",
        "支払利息等", "動産不動産賃借料", "租税公課", "期中平均役員数_人", "期中平均従業員数_人", "付加価値", "営業純益"]),
    "houjin_table1_all_ex_fin_cost_structure_FY1980_2025.csv": ("all", 1980, [
        "売上高", "売上原価", "販売費及び一般管理費", "営業利益"]),
}


def normalize(label):
    """項目名から、単位・（当期末）・空白・脚注記号を取り除く。"""
    s = str(label)
    s = re.sub(r"[\s　]+", "", s)
    s = re.sub(r"【[^】]*】", "", s)
    s = re.sub(r"[（(]当期末[）)]", "", s)
    s = s.replace("※", "")
    return s


def to_number(x):
    s = str(x).strip().replace(",", "").replace("，", "")
    if s in ("", "***", "*", "-", "－", "…", "x", "X", "nan", "None"):
        return float("nan")
    try:
        return float(s)
    except ValueError:
        return float("nan")


def read_rows(path):
    """CSV・Excelを、文字列の行のリストにして返す。"""
    path = Path(path)
    if path.suffix.lower() in (".xlsx", ".xlsm"):
        import openpyxl
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            wb = openpyxl.load_workbook(path, data_only=True)  # read_only は、e-Statのxlsxの範囲情報が不完全で、行を取りこぼすので使わない
        ws = wb[wb.sheetnames[0]]
        rows = []
        for r in ws.iter_rows(values_only=True):
            rows.append(["" if v is None else (str(int(v)) if isinstance(v, float) and v == int(v) else str(v)) for v in r])
        return rows
    raw = path.read_bytes()
    for enc in ("utf-8-sig", "cp932"):
        try:
            text = raw.decode(enc)
            break
        except UnicodeDecodeError:
            continue
    else:
        raise ValueError(f"{path}: 文字コードを判定できません")
    return [r for r in csv.reader(io.StringIO(text))]


YEAR = re.compile(r"^\s*(\d{4})\s*年?度?\s*$")
YEAR_STRICT = re.compile(r"^\s*(\d{4})\s*年度\s*$")


def read_estat_table(path):
    """行＝年度、列＝正規化した項目名の表を返す。"""
    rows = read_rows(path)
    # (1) e-StatのCSV（縦持ち）：cat01_code, 調査項目, cat02_code, 業種, cat03_code, 規模, time_code, 年度, unit, value
    for i, r in enumerate(rows):
        if "value" in r and any(str(c).startswith("調査項目") for c in r):
            col_item = next(j for j, c in enumerate(r) if str(c).startswith("調査項目"))
            col_time = next(j for j, c in enumerate(r) if re.sub(r"[\s　]+", "", str(c)) in ("年期", "年度", "時間軸"))
            col_val = r.index("value")
            data = {}
            for rr in rows[i + 1:]:
                if len(rr) <= max(col_item, col_time, col_val):
                    continue
                m = YEAR.match(rr[col_time])
                if not m:
                    continue
                name = normalize(rr[col_item])
                if not name:
                    continue
                data.setdefault(int(m.group(1)), {}).setdefault(name, to_number(rr[col_val]))  # 同名の項目は最初のものを使う
            df = pd.DataFrame.from_dict(data, orient="index").sort_index()
            df.index.name = "年度"
            return df
    # (2) Excel・行が年度のCSV：項目名の並ぶ行があり、データ行の先頭付近に「2009年度」のような年度がある
    for i, r in enumerate(rows):
        names = [normalize(c) for c in r]
        labels = [n for n in names if n and not re.fullmatch(r"[0-9,.\-]+", n)]
        if "売上高" in names and len(labels) >= 5:
            data = {}
            for rr in rows[i + 1:]:
                year = None
                for c in rr[:4]:
                    m = YEAR_STRICT.match(c)
                    if m:
                        year = int(m.group(1))
                        break
                if year is None and rr and YEAR.match(rr[0]):
                    year = int(YEAR.match(rr[0]).group(1))
                if year is None:
                    continue
                data[year] = {}
                for n, v in zip(names, rr):
                    if n and n not in data[year] and not YEAR.match(str(v)):
                        data[year][n] = to_number(v)
            df = pd.DataFrame.from_dict(data, orient="index").sort_index()
            df.index.name = "年度"
            return df
    # (3) 行が項目・列が年度のCSV
    for i, r in enumerate(rows):
        years = [(j, YEAR.match(c)) for j, c in enumerate(r)]
        years = [(j, int(m.group(1))) for j, m in years if m]
        if len(years) >= 10:
            data = {}
            for rr in rows[i + 1:]:
                if not rr or not normalize(rr[0]):
                    continue
                name = normalize(rr[0])
                data[name] = {y: to_number(rr[j]) for j, y in years if j < len(rr)}
            df = pd.DataFrame(data).sort_index()
            df.index.name = "年度"
            return df
    raise ValueError(f"{path}: 年度と項目（売上高・経常利益など）の表を見つけられませんでした")


def classify(path):
    """ファイルの中の「業種」「規模」の値から、どの表かを判定する（ファイル名は問わない）。判定できなければ None。"""
    try:
        rows = read_rows(path)
    except Exception:  # noqa: BLE001
        return None
    # Excel：先頭付近に「業種 (…)：、コード、名前」「規模 (…)：、コード、名前」の行がある
    industry = size = None
    for r in rows[:15]:
        if r and str(r[0]).startswith("業種"):
            vals = [c for c in r[1:] if str(c).strip()]
            industry = normalize(vals[-1]) if vals else None
        if r and str(r[0]).startswith("規模"):
            vals = [c for c in r[1:] if str(c).strip()]
            size = normalize(vals[-1]) if vals else None
    if industry and size:
        return SELECT.get((industry, size))
    # CSV（縦持ち）：「業種」「規模」の列の値がそれぞれ1種類
    for i, r in enumerate(rows):
        if "value" in r and any(str(c).startswith("調査項目") for c in r):
            ci = next((j for j, c in enumerate(r) if str(c).startswith("業種")), None)
            cs = next((j for j, c in enumerate(r) if str(c).startswith("規模")), None)
            if ci is None or cs is None:
                return None
            body = [x for x in rows[i + 1:i + 5000] if len(x) > max(ci, cs)]
            inds = {normalize(x[ci]) for x in body}
            sizes = {normalize(x[cs]) for x in body}
            if len(inds) == 1 and len(sizes) == 1:
                return SELECT.get((next(iter(inds)), next(iter(sizes))))
            return None
    return None


def pick(df, col):
    name = ITEM[col]
    if name not in df.columns:
        raise KeyError(f"項目「{name}」が見つかりません。e-Statの表で、全項目・全年度を選んでダウンロードしたか確認してください。")
    return df[name]


def fmt(v):
    return "" if pd.isna(v) else str(int(round(v)))


def main():
    found = {}
    for f in sorted(list(ESTAT.glob("*.csv")) + list(ESTAT.glob("*.xlsx"))):
        if f.name.startswith("~$"):
            continue
        k = classify(f)
        if k:
            found[k] = f
    for key, fname in INPUTS.items():  # 決まった名前のファイルがあれば、それを優先
        if (ESTAT / fname).exists():
            found[key] = ESTAT / fname
    missing = [k for k in INPUTS if k not in found]
    if missing:
        sys.exit("e-Statのファイルが足りません。data/estat/ に、次の表のファイルを置いてください（ファイル名は自由）:\n"
                 + "\n".join(f"  - {DESC[k]}" for k in missing))
    tables = {}
    for key, path in found.items():
        tables[key] = read_estat_table(path)
        print(f"OK  {path.name}（{DESC[key]}）: {tables[key].index.min()}〜{tables[key].index.max()}年度、{tables[key].shape[1]}項目")
    RAW.mkdir(parents=True, exist_ok=True)
    for out, (key, first, cols) in OUTPUTS.items():
        df = tables[key]
        df = df[df.index >= first]
        lines = ["年度," + ",".join(cols)]
        for y, row in df.iterrows():
            vals = [fmt(pick(df, c)[y]) for c in cols]
            lines.append(f"{y}," + ",".join(vals))
        (RAW / out).write_text("\n".join(lines) + "\n", encoding="utf-8")
        print(f"  → data/raw/{out}（{len(df)}年度）")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile build_data.py
"""企業の最高益の分解：取得済み生データから分析用CSVを作る。

入力は data/raw/ にある次のファイル。出力は data/ に書く。パスはこのファイルからの相対。

- houjin_table1_all_ex_fin_FY1980_2025.csv   法人企業統計 全産業（除く金融保険業）・全規模・年度
- houjin_table1_pure_holding_FY2009_2025.csv  同 純粋持株会社
- keitunenenpo.xlsx                           財務省公表の「純粋持株会社を除く経常利益」（検算用）
- boj_bp_direct_investment_monthly.csv        日銀 国際収支統計(6版) 月次（億円、Shift_JIS）
- boj_fx_usdjpy_monthly_avg.csv               日銀 東京市場ドル・円 17時 月中平均（Shift_JIS）
- 2024i2_jp.xlsx                              内閣府 国民経済計算 2024年度年次推計 非金融法人企業

実行: python build_data.py
"""
import csv
import hashlib
import io
import json
import unicodedata
from pathlib import Path

import openpyxl
import pandas as pd

HERE = Path(__file__).resolve().parent
RAW = HERE / "data" / "raw"
OUT = HERE / "data"


def read_boj(path):
    text = Path(path).read_bytes().decode("cp932")
    lines = text.splitlines()
    start = next(i for i, l in enumerate(lines) if l.startswith("SERIES_CODE,"))
    df = pd.read_csv(io.StringIO("\n".join(lines[start:])))
    df["date"] = df["SURVEY_DATES"].astype(str)
    df["year"] = df["date"].str[:4].astype(int)
    df["month"] = df["date"].str[4:6].astype(int)
    df["fy"] = df["year"].where(df["month"] >= 4, df["year"] - 1)
    return df


# ---- 1. 国際収支（直接投資収益など）：暦年・年度に集計 ----
bp = read_boj(RAW / "boj_bp_direct_investment_monthly.csv")
names = {
    "BPBP6JYNPIN": "第一次所得収支_ネット",
    "BPBP6JYNPIC": "第一次所得_受取",
    "BPBP6JYNPIC2": "投資収益_受取",
    "BPBP6JYNPIC21": "直接投資収益_受取",
    "BPBP6JYNPID21": "直接投資収益_支払",
    "BPBP6JYNPIC211": "直接投資_出資所得_受取",
    "BPBP6JYNPIC2111": "直接投資_配当金配分済支店収益_受取",
    "BPBP6JYNPIC2112": "直接投資_再投資収益_受取",
    "BPBP6JYNPIC212": "直接投資_利子所得_受取",
    "BPBP6JYNPIC22": "証券投資収益_受取",
}
bp["系列"] = bp["SERIES_CODE"].map(names)
bp["VALUES"] = pd.to_numeric(bp["VALUES"], errors="coerce")


def agg(df, key, months_needed=12):
    g = df.groupby([key, "系列"])["VALUES"].agg(["sum", "count"]).reset_index()
    g = g[g["count"] == months_needed]  # 12か月そろった年だけ
    return g.pivot(index=key, columns="系列", values="sum")[list(names.values())]


bp_fy = agg(bp, "fy")
bp_fy.index.name = "年度"
bp_fy.to_csv(OUT / "bop_fy_oku_yen.csv", encoding="utf-8-sig")
bp_cy = agg(bp, "year")
bp_cy.index.name = "暦年"
bp_cy.to_csv(OUT / "bop_cy_oku_yen.csv", encoding="utf-8-sig")

# ---- 2. 為替：ドル円 月中平均を年度・暦年平均に ----
fx = read_boj(RAW / "boj_fx_usdjpy_monthly_avg.csv")
fx["VALUES"] = pd.to_numeric(fx["VALUES"], errors="coerce")
fx_fy = fx.groupby("fy")["VALUES"].agg(["mean", "count"])
fx_fy = fx_fy[fx_fy["count"] == 12][["mean"]].rename(columns={"mean": "ドル円_年度平均"})
fx_fy.index.name = "年度"
fx_fy.round(4).to_csv(OUT / "usdjpy_fy_avg.csv", encoding="utf-8-sig")

# ---- 2b. 物価：国内企業物価指数（財）と企業向けサービス価格指数（サービス）の年度平均 ----
def fy_mean(path):
    df = read_boj(path)
    df["VALUES"] = pd.to_numeric(df["VALUES"], errors="coerce")
    g = df.groupby("fy")["VALUES"].agg(["mean", "count"])
    return g[g["count"] == 12]["mean"]


defl = pd.DataFrame({"国内企業物価指数_総平均": fy_mean(RAW / "boj_cgpi_monthly.csv"),
                     "企業向けサービス価格指数_総平均": fy_mean(RAW / "boj_sppi_monthly.csv")})
defl.index.name = "年度"
defl.round(4).to_csv(OUT / "deflators_fy.csv", encoding="utf-8-sig")

# ---- 3. SNA 非金融法人企業（暦年、10億円） ----
def sna_rows(path, sector):
    ws = openpyxl.load_workbook(path, data_only=True)["暦年（１）"]
    years = [c.value for c in ws[7][1:] if c.value is not None]
    want = {
        "営業余剰_純": "1.3",
        "営業余剰_総": "（再掲）営業余剰（総）",
        "固定資本減耗": "（控除）固定資本減耗",
        "財産所得_受取": "1.4",
        "利子_受取": None,
        "配当_受取": None,
        "海外直接投資再投資収益_受取": None,
        "財産所得_支払": "1.1",
        "第1次所得バランス_純": "1.2",
        "第1次所得バランス_総": "（再掲）第１次所得バランス（総）",
    }
    rows = {}
    section = None  # 受取側／支払側の見出し位置で（１）利子などを区別する
    for r in ws.iter_rows(min_row=8, values_only=True):
        label = unicodedata.normalize("NFKC", str(r[0] or "")).replace("　", "").replace(" ", "")
        if label.startswith("1.1"):
            section = "支払"
        elif label.startswith("1.4"):
            section = "受取"
        vals = list(r[1:1 + len(years)])
        if label.startswith("1.3"):
            rows["営業余剰_純"] = vals
        elif label.startswith("(再掲)営業余剰(総)"):
            rows["営業余剰_総"] = vals
        elif label.startswith("(控除)固定資本減耗") and "固定資本減耗" not in rows:
            rows["固定資本減耗"] = vals
        elif label.startswith("1.4"):
            rows["財産所得_受取"] = vals
        elif label.startswith("1.1"):
            rows["財産所得_支払"] = vals
        elif label.startswith("1.2"):
            rows["第1次所得バランス_純"] = vals
        elif label.startswith("(再掲)第1次所得バランス(総)"):
            rows["第1次所得バランス_総"] = vals
        elif section == "受取" and label.startswith("(1)利子"):
            rows["利子_受取"] = vals
        elif section == "受取" and label.startswith("a.配当"):
            rows["配当_受取"] = vals
        elif section == "受取" and label.startswith("(3)海外直接投資に関する再投資収益"):
            rows["海外直接投資再投資収益_受取"] = vals
        elif section == "支払" and label.startswith("(1)利子"):
            rows["利子_支払"] = vals
        elif section == "支払" and label.startswith("a.配当"):
            rows["配当_支払"] = vals
    df = pd.DataFrame(rows, index=[int(y) for y in years])
    df.index.name = "暦年"
    df.insert(0, "部門", sector)
    return df


sna_nf = sna_rows(RAW / "2024i2_jp.xlsx", "非金融法人企業")
sna_nf.to_csv(OUT / "sna_nonfin_corp_cy_10oku_yen.csv", encoding="utf-8-sig")

# ---- 4. 法人企業統計 ----
h = pd.read_csv(RAW / "houjin_table1_all_ex_fin_FY1980_2025.csv").set_index("年度")
ph = pd.read_csv(RAW / "houjin_table1_pure_holding_FY2009_2025.csv").set_index("年度")
# 恒等式：経常利益 = 営業利益 + 営業外収益 - 営業外費用
assert (h["営業利益"] + h["営業外収益"] - h["営業外費用"] - h["経常利益"]).abs().max() == 0
# 財務省公表値（純粋持株会社を除く経常利益）との突き合わせ
ws = openpyxl.load_workbook(RAW / "keitunenenpo.xlsx", data_only=True).active
mof = {}
for r in ws.iter_rows(values_only=True):
    if r[0] and str(r[0]).startswith(("平成", "令和")):
        s = unicodedata.normalize("NFKC", r[0]).replace("元", "1")
        n = int("".join(c for c in s if c.isdigit()))
        mof[1988 + n if s.startswith("平成") else 2018 + n] = (r[1] * 100, r[2] * 100, r[3] * 100)
diff = max(
    max(abs(v[0] - h.loc[y, "経常利益"]), abs(v[1] - ph.loc[y, "経常利益"])) for y, v in mof.items()
)
assert diff < 1, diff
print("財務省公表値との最大差（百万円）:", diff)

# ---- 5. 結合表（年度ベース） ----
m = h[["売上高", "営業利益", "営業外収益", "営業外費用", "経常利益", "支払利息等", "配当金計", "当期純利益"]].copy()
m = m.join(ph[["経常利益", "営業利益", "営業外収益"]].add_prefix("純粋持株会社_"))
m["経常利益_純粋持株会社を除く"] = m["経常利益"] - m["純粋持株会社_経常利益"].fillna(0)
m = m.join(bp_fy).join(fx_fy)
m.to_csv(OUT / "merged_fy_million_yen_and_oku_yen.csv", encoding="utf-8-sig")

# ---- 6. 取得元の記録（SHA-256付き） ----
def sha(p):
    p = Path(p)
    return hashlib.sha256(p.read_bytes()).hexdigest() if p.exists() else None


manifest = {
    "houjin_table1_all_ex_fin": {
        "file": "raw/houjin_table1_all_ex_fin_FY1980_2025.csv",
        "source": "法人企業統計調査 時系列データ 表1 金融業、保険業以外の業種（原数値、年度）。全産業（除く金融保険業）・全規模",
        "url": "https://www.e-stat.go.jp/dbview?sid=0003060791",
        "retrieved": "2026-10-10",
        "method": "e-Statの画面からダウンロードしたCSV（README参照）から parse_estat.py で作成。FY2025（2025年度）まで",
        "sha256": sha(RAW / "houjin_table1_all_ex_fin_FY1980_2025.csv"),
        "unit": "百万円（母集団は社、従業員は人）",
    },
    "houjin_table1_pure_holding": {
        "file": "raw/houjin_table1_pure_holding_FY2009_2025.csv",
        "source": "同表 業種=純粋持株会社・全規模",
        "url": "https://www.e-stat.go.jp/dbview?sid=0003060791",
        "retrieved": "2026-10-10",
        "sha256": sha(RAW / "houjin_table1_pure_holding_FY2009_2025.csv"),
        "unit": "百万円",
    },
    "houjin_value_added_all": {
        "file": "raw/houjin_table1_all_ex_fin_value_added_FY1980_2025.csv",
        "source": "同表 全産業（除く金融保険業）・全規模：付加価値と内訳（人件費、支払利息等、賃借料、租税公課、営業純益）、従業員数・役員数",
        "url": "https://www.e-stat.go.jp/dbview?sid=0003060791",
        "retrieved": "2026-10-10",
        "method": "e-Statの画面からダウンロードしたCSVから作成。付加価値＝人件費＋支払利息等＋動産・不動産賃借料＋租税公課＋営業純益が全年で成立（役員賞与はFY2006まで人件費に含まれない）",
        "sha256": sha(RAW / "houjin_table1_all_ex_fin_value_added_FY1980_2025.csv"),
        "unit": "百万円（人数は人）",
    },
    "houjin_value_added_10oku_plus": {
        "file": "raw/houjin_table1_capital_10oku_plus_value_added_FY1980_2025.csv",
        "source": "同表 資本金10億円以上（全産業・除く金融保険業）。10億円未満は全規模との差で求める",
        "url": "https://www.e-stat.go.jp/dbview?sid=0003060791",
        "retrieved": "2026-10-10",
        "sha256": sha(RAW / "houjin_table1_capital_10oku_plus_value_added_FY1980_2025.csv"),
        "unit": "百万円（人数は人）",
    },
    "boj_cgpi": {
        "file": "raw/boj_cgpi_monthly.csv",
        "source": "日本銀行 DB=PR01 PRCG20_2200000000 国内企業物価指数 総平均（2020年基準）月次",
        "url": "https://www.stat-search.boj.or.jp/api/v1/getDataCode?format=csv&lang=jp&db=PR01&startDate=198001&endDate=202608&code=PRCG20_2200000000",
        "retrieved": "2026-10-10",
        "sha256": sha(RAW / "boj_cgpi_monthly.csv"),
        "unit": "2020年=100",
    },
    "boj_sppi": {
        "file": "raw/boj_sppi_monthly.csv",
        "source": "日本銀行 DB=PR02 PRCS20_5200000000 企業向けサービス価格指数 基本分類指数 総平均（2020年基準）月次",
        "url": "https://www.stat-search.boj.or.jp/api/v1/getDataCode?format=csv&lang=jp&db=PR02&startDate=198501&endDate=202608&code=PRCS20_5200000000",
        "retrieved": "2026-10-10",
        "sha256": sha(RAW / "boj_sppi_monthly.csv"),
        "unit": "2020年=100",
    },
    "houjin_cost_structure": {
        "file": "raw/houjin_table1_all_ex_fin_cost_structure_FY1980_2025.csv",
        "source": "同表 全産業（除く金融保険業）・全規模：売上高、売上原価、販売費及び一般管理費、営業利益",
        "url": "https://www.e-stat.go.jp/dbview?sid=0003060791",
        "retrieved": "2026-10-10",
        "method": "画面上の表を抽出。列合計を画面側と突き合わせて確認。売上高−売上原価−販管費＝営業利益が全年で成立",
        "sha256": sha(RAW / "houjin_table1_all_ex_fin_cost_structure_FY1980_2025.csv"),
        "unit": "百万円",
    },
    "mof_keitunenenpo": {
        "file": "raw/keitunenenpo.xlsx",
        "source": "財務省 財務総合政策研究所『純粋持株会社を除く経常利益』（年度）",
        "url": "https://www.mof.go.jp/pri/reference/ssc/results/keitunenenpo.xlsx",
        "retrieved": "2026-10-10",
        "sha256": sha(RAW / "keitunenenpo.xlsx"),
        "unit": "億円",
    },
    "boj_bp": {
        "file": "raw/boj_bp_direct_investment_monthly.csv",
        "source": "日本銀行 時系列統計データ検索サイト（API）DB=BP01 国際収支統計（6版基準）月次、第一次所得関連",
        "url": "https://www.stat-search.boj.or.jp/api/v1/getDataCode?format=csv&lang=jp&db=BP01&startDate=199601&endDate=202608&code=" + ",".join(names),
        "retrieved": "2026-10-10",
        "sha256": sha(RAW / "boj_bp_direct_investment_monthly.csv"),
        "unit": "億円",
    },
    "boj_fx": {
        "file": "raw/boj_fx_usdjpy_monthly_avg.csv",
        "source": "日本銀行 DB=FM08 FXERM07 東京市場ドル・円スポット17時時点／月中平均",
        "url": "https://www.stat-search.boj.or.jp/api/v1/getDataCode?format=csv&lang=jp&db=FM08&startDate=199601&endDate=202609&code=FXERM07",
        "retrieved": "2026-10-10",
        "sha256": sha(RAW / "boj_fx_usdjpy_monthly_avg.csv"),
        "unit": "1ドルにつき円",
    },
    "sna_nonfin_corp": {
        "file": "raw/2024i2_jp.xlsx",
        "source": "内閣府 国民経済計算 2024年度年次推計（2020年基準）制度部門別所得支出勘定 非金融法人企業",
        "url": "https://www.esri.cao.go.jp/jp/sna/data/data_list/kakuhou/files/2024/tables/2024i2_jp.xlsx",
        "retrieved": "2026-09-09（交易損失②で取得、SHA-256一致を確認して複製）",
        "sha256": sha(RAW / "2024i2_jp.xlsx"),
        "unit": "10億円",
    },
}
(OUT / "source_manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=1), encoding="utf-8")
print("done:", sorted(p.name for p in OUT.glob("*.csv")))

In [ ]:
%%writefile analyze.py
"""企業の最高益の分解。build_data.py の出力から、記事の数値・図の元データを計算する。

実行: python analyze.py   （先に build_data.py を実行しておく）
出力: data/decomposition.csv, data/fx_counterfactual.csv, data/sna_vs_houjin.csv, data/metrics.json
"""
import json
from pathlib import Path

import pandas as pd

HERE = Path(__file__).resolve().parent
D = HERE / "data"
RAW = D / "raw"
MIL = 1e6  # 百万円 -> 兆円

m = pd.read_csv(D / "merged_fy_million_yen_and_oku_yen.csv", index_col=0)
h = pd.read_csv(RAW / "houjin_table1_all_ex_fin_FY1980_2025.csv").set_index("年度")
ph = pd.read_csv(RAW / "houjin_table1_pure_holding_FY2009_2025.csv").set_index("年度")
sna = pd.read_csv(D / "sna_nonfin_corp_cy_10oku_yen.csv", index_col=0)

BASE, LAST = 2012, 2025
metrics = {"base_year": BASE, "last_year": LAST}


def decomp(df, a, b):
    d = df.loc[b] - df.loc[a]
    tot = d["経常利益"]
    out = {
        "経常利益_始点_兆円": df.loc[a, "経常利益"] / MIL,
        "経常利益_終点_兆円": df.loc[b, "経常利益"] / MIL,
        "増分_兆円": tot / MIL,
        "営業利益_兆円": d["営業利益"] / MIL,
        "営業外収益_兆円": d["営業外収益"] / MIL,
        "営業外費用の増加_兆円": d["営業外費用"] / MIL,  # 経常利益から差し引く向き
    }
    out["営業利益_割合"] = d["営業利益"] / tot
    out["営業外収益_割合"] = d["営業外収益"] / tot
    out["営業外費用_割合"] = -d["営業外費用"] / tot
    assert abs(d["営業利益"] + d["営業外収益"] - d["営業外費用"] - tot) < 1
    return out


# ---- 1. 経常利益の増分の分解（全産業・除く金融保険業） ----
rows = {}
for a, b in [(2012, 2025), (2012, 2024), (2000, 2025), (2013, 2025)]:
    rows[f"{a}->{b}"] = decomp(h, a, b)
# 純粋持株会社を除く（営業利益・営業外損益も除ける。FY2009〜）
hx = h.loc[ph.index, ["営業利益", "営業外収益", "営業外費用", "経常利益"]] - ph[["営業利益", "営業外収益", "営業外費用", "経常利益"]]
for a, b in [(2012, 2025)]:
    rows[f"{a}->{b}_持株会社除く"] = decomp(hx, a, b)
metrics["decomposition"] = rows
pd.DataFrame(rows).T.to_csv(D / "decomposition.csv", encoding="utf-8-sig")

# ---- 2. 営業利益の増分：売上高の増加 と 利益率の上昇 ----
s0, s1 = h.loc[BASE, "売上高"], h.loc[LAST, "売上高"]
m0, m1 = h.loc[BASE, "営業利益"] / s0, h.loc[LAST, "営業利益"] / s1
sales_eff = (s1 - s0) * m0
margin_eff = s1 * (m1 - m0)
assert abs(sales_eff + margin_eff - (h.loc[LAST, "営業利益"] - h.loc[BASE, "営業利益"])) < 1
metrics["operating_profit_split"] = {
    "売上高_始点_兆円": s0 / MIL, "売上高_終点_兆円": s1 / MIL, "売上高_増加率": s1 / s0 - 1,
    "営業利益率_始点": m0, "営業利益率_終点": m1,
    "売上増の効果_兆円": sales_eff / MIL, "利益率上昇の効果_兆円": margin_eff / MIL,
    "利益率上昇の割合": margin_eff / (sales_eff + margin_eff),
}
# 分け方は計算の順序で変わる。売上を先に見る（始点の利益率で評価）／利益率を先に見る（始点の売上で評価）の二通りと、その平均
_dS, _dm = s1 - s0, m1 - m0
_tot = h.loc[LAST, "営業利益"] - h.loc[BASE, "営業利益"]
metrics["operating_profit_split"].update({
    "売上高の増加倍率": s1 / s0, "営業利益率の上昇倍率": m1 / m0, "営業利益の増加倍率": h.loc[LAST, "営業利益"] / h.loc[BASE, "営業利益"],
    "売上増の効果_始点の利益率で評価_兆円": _dS * m0 / MIL, "売上増の効果_終点の利益率で評価_兆円": _dS * m1 / MIL,
    "利益率上昇の効果_始点の売上で評価_兆円": s0 * _dm / MIL, "利益率上昇の効果_終点の売上で評価_兆円": s1 * _dm / MIL,
    "交差項_兆円": _dS * _dm / MIL,
    "利益率上昇の割合_下限": s0 * _dm / _tot, "利益率上昇の割合_上限": s1 * _dm / _tot,
    "売上増の効果_平均_兆円": (_dS * m0 + _dS * m1) / 2 / MIL, "利益率上昇の効果_平均_兆円": (s0 * _dm + s1 * _dm) / 2 / MIL,
    "売上増の割合_平均": (_dS * m0 + _dS * m1) / 2 / _tot, "利益率上昇の割合_平均": (s0 * _dm + s1 * _dm) / 2 / _tot,
})
# 参考：営業利益率の推移
metrics["営業利益率_系列"] = (h["営業利益"] / h["売上高"]).round(5).to_dict()

# ---- 3. 営業外収益と国際収支（直接投資収益） ----
bp = m[["直接投資収益_受取", "直接投資_配当金配分済支店収益_受取", "直接投資_再投資収益_受取"]] / 1e4  # 億円 -> 兆円
nonop = m["営業外収益"] / MIL
metrics["nonop_vs_bop"] = {
    "営業外収益_増分_兆円": nonop[LAST] - nonop[BASE],
    "直接投資収益_受取_増分_兆円": bp.loc[LAST, "直接投資収益_受取"] - bp.loc[BASE, "直接投資収益_受取"],
    "うち配当金配分済支店収益_増分_兆円": bp.loc[LAST, "直接投資_配当金配分済支店収益_受取"] - bp.loc[BASE, "直接投資_配当金配分済支店収益_受取"],
    "うち再投資収益_増分_兆円": bp.loc[LAST, "直接投資_再投資収益_受取"] - bp.loc[BASE, "直接投資_再投資収益_受取"],
}
metrics["nonop_vs_bop"]["配当金配分済支店収益の増分／営業外収益の増分"] = (
    metrics["nonop_vs_bop"]["うち配当金配分済支店収益_増分_兆円"] / metrics["nonop_vs_bop"]["営業外収益_増分_兆円"]
)
# 参考：外国の株式・債券などからの配当・利子（証券投資収益）。金融機関・年金・家計などの受取を含み、法人企業統計の対象（金融業・保険業を除く企業）にどれだけ入るかは分からない
_sec = m["証券投資収益_受取"] / 1e4
metrics["nonop_vs_bop"]["証券投資収益_受取_2012_兆円"] = _sec[BASE]
metrics["nonop_vs_bop"]["証券投資収益_受取_2025_兆円"] = _sec[LAST]
metrics["nonop_vs_bop"]["証券投資収益_増分_兆円"] = _sec[LAST] - _sec[BASE]
metrics["nonop_vs_bop"]["直接投資の配当金・利子と証券投資収益の増分合計_兆円"] = (
    metrics["nonop_vs_bop"]["うち配当金配分済支店収益_増分_兆円"]
    + (bp_int := m["直接投資_利子所得_受取"] / 1e4)[LAST] - bp_int[BASE]
    + metrics["nonop_vs_bop"]["証券投資収益_増分_兆円"]
)

# ---- 4. 円安の寄与の目安（直接投資収益の全額がドル建てと仮定。上限の目安） ----
fx = m["ドル円_年度平均"]
cf = pd.DataFrame(index=m.index[m.index >= BASE])
cf["ドル円"] = fx
cf["直接投資収益_受取_兆円"] = bp["直接投資収益_受取"]
cf["円レート固定_兆円"] = cf["直接投資収益_受取_兆円"] * fx[BASE] / fx
cf["円安の寄与_兆円"] = cf["直接投資収益_受取_兆円"] - cf["円レート固定_兆円"]
cf = cf.dropna()
cf.round(4).to_csv(D / "fx_counterfactual.csv", encoding="utf-8-sig")
metrics["fx_counterfactual"] = {
    "基準レート_円": fx[BASE], "最終年レート_円": fx[LAST],
    "最終年の円安の寄与_兆円": cf.loc[LAST, "円安の寄与_兆円"],
    "経常利益増分に対する割合_上限": cf.loc[LAST, "円安の寄与_兆円"] / metrics["decomposition"]["2012->2025"]["増分_兆円"],
    "換算差の内訳_配当金配分済支店収益_兆円": bp.loc[LAST, "直接投資_配当金配分済支店収益_受取"] * (1 - fx[BASE] / fx[LAST]),
    "換算差の内訳_再投資収益_兆円": bp.loc[LAST, "直接投資_再投資収益_受取"] * (1 - fx[BASE] / fx[LAST]),
    "換算差の内訳_利子所得_兆円": (m["直接投資_利子所得_受取"] / 1e4)[LAST] * (1 - fx[BASE] / fx[LAST]),
    "反実仮想_2012年度の為替での2025年度の直接投資収益_兆円": cf.loc[LAST, "円レート固定_兆円"],
    "前提": "直接投資収益（受取）の全額がドル建てと仮定し、FY2012のレートに固定して換算し直した差（シナリオ計算で、上限ではない。ドル以外の通貨の動きは含まない）。再投資収益の分は法人単体の営業外収益に現れない。営業利益に含まれる輸出採算の円安効果は含まない",
}

# ---- 5. 支払利息等 ----
ie = h["支払利息等"] / MIL
metrics["interest"] = {
    "支払利息等_1995_兆円": ie[1995], "支払利息等_2000_兆円": ie[2000], "支払利息等_2012_兆円": ie[2012],
    "支払利息等_2021_兆円": ie[2021], "支払利息等_2025_兆円": ie[LAST],
    "支払利息等／営業利益_1995": h.loc[1995, "支払利息等"] / h.loc[1995, "営業利益"],
    "支払利息等／営業利益_2012": h.loc[2012, "支払利息等"] / h.loc[2012, "営業利益"],
    "支払利息等／営業利益_2025": h.loc[LAST, "支払利息等"] / h.loc[LAST, "営業利益"],
    "支払利息等の減少が経常利益を押し上げた額_1995から2012_兆円": ie[1995] - ie[2012],
    "支払利息等の増加が経常利益を押し下げた額_2012から2025_兆円": ie[LAST] - ie[2012],
}

# ---- 6. SNAとの突き合わせ（年度tを暦年tに対応させる。3か月ずれる） ----
yrs = [y for y in sna.index if y in h.index]
cmp_ = pd.DataFrame(index=yrs)
cmp_["法人企業統計_営業利益_兆円"] = h.loc[yrs, "営業利益"] / MIL
cmp_["法人企業統計_営業利益＋減価償却費_兆円"] = (h.loc[yrs, "営業利益"] + h.loc[yrs, "減価償却費計"]) / MIL
cmp_["SNA_営業余剰_総_兆円"] = sna.loc[yrs, "営業余剰_総"] / 1e3
cmp_["SNA_営業余剰_純_兆円"] = sna.loc[yrs, "営業余剰_純"] / 1e3
cmp_["SNA_固定資本減耗_兆円"] = sna.loc[yrs, "固定資本減耗"] / 1e3
cmp_["法人企業統計_減価償却費_兆円"] = h.loc[yrs, "減価償却費計"] / MIL
cmp_["SNA_財産所得_受取_兆円"] = sna.loc[yrs, "財産所得_受取"] / 1e3
cmp_.index.name = "年"
cmp_.round(3).to_csv(D / "sna_vs_houjin.csv", encoding="utf-8-sig")
c94, c24 = cmp_.loc[1994], cmp_.loc[2024]
metrics["sna_vs_houjin"] = {
    "法人_営業利益_1994": c94.iloc[0], "法人_営業利益_2024": c24.iloc[0],
    "SNA_営業余剰純_1994": c94["SNA_営業余剰_純_兆円"], "SNA_営業余剰純_2024": c24["SNA_営業余剰_純_兆円"],
    "SNA_営業余剰総_1994": c94["SNA_営業余剰_総_兆円"], "SNA_営業余剰総_2024": c24["SNA_営業余剰_総_兆円"],
    "SNA_固定資本減耗_1994": c94["SNA_固定資本減耗_兆円"], "SNA_固定資本減耗_2024": c24["SNA_固定資本減耗_兆円"],
    "法人_減価償却費_1994": c94["法人企業統計_減価償却費_兆円"], "法人_減価償却費_2024": c24["法人企業統計_減価償却費_兆円"],
}

# 純粋持株会社
metrics["pure_holding"] = {
    "経常利益シェア_2012": ph.loc[2012, "経常利益"] / h.loc[2012, "経常利益"],
    "経常利益シェア_2025": ph.loc[LAST, "経常利益"] / h.loc[LAST, "経常利益"],
}


# ---- 7. 付加価値の分配と労働分配率 ----
def load_va(name):
    v = pd.read_csv(RAW / name).set_index("年度").fillna(0)
    # 人件費：役員給与＋従業員給与＋従業員賞与＋福利厚生費。役員賞与はFY2007から含める（表の付加価値の内訳と一致させるため）
    v["人件費"] = v["役員給与"] + v["従業員給与"] + v["従業員賞与"] + v["福利厚生費"] + v["役員賞与"] * (v.index >= 2007)
    v["人数"] = v["期中平均従業員数_人"] + v["期中平均役員数_人"]
    return v


va_all = load_va("houjin_table1_all_ex_fin_value_added_FY1980_2025.csv")
va_big = load_va("houjin_table1_capital_10oku_plus_value_added_FY1980_2025.csv")
cols = ["付加価値", "人件費", "営業純益", "支払利息等", "動産不動産賃借料", "租税公課", "人数"]
va_small = va_all[cols] - va_big[cols]
for v in (va_all, va_big):  # 恒等式：付加価値＝人件費＋支払利息等＋賃借料＋租税公課＋営業純益
    res = v["付加価値"] - (v["人件費"] + v["支払利息等"] + v["動産不動産賃借料"] + v["租税公課"] + v["営業純益"])
    assert res.abs().max() == 0
    assert (v["営業利益"] - v["支払利息等"] - v["営業純益"]).abs().max() == 0

ls = pd.DataFrame({
    "全規模": va_all["人件費"] / va_all["付加価値"],
    "資本金10億円以上": va_big["人件費"] / va_big["付加価値"],
    "資本金10億円未満": va_small["人件費"] / va_small["付加価値"],
})
ls.index.name = "年度"
ls.round(5).to_csv(D / "labor_share_by_size.csv", encoding="utf-8-sig")

dist = {}
for a_, b_ in [(2012, 2025), (2000, 2025)]:
    d = va_all.loc[b_] - va_all.loc[a_]
    dist[f"{a_}->{b_}"] = {
        "付加価値_始点_兆円": va_all.loc[a_, "付加価値"] / MIL, "付加価値_終点_兆円": va_all.loc[b_, "付加価値"] / MIL,
        "付加価値_増分_兆円": d["付加価値"] / MIL,
        **{f"{k}_兆円": d[k] / MIL for k in ["人件費", "営業純益", "動産不動産賃借料", "租税公課", "支払利息等"]},
        **{f"{k}_割合": d[k] / d["付加価値"] for k in ["人件費", "営業純益", "動産不動産賃借料", "租税公課", "支払利息等"]},
        "労働分配率_始点": ls.loc[a_, "全規模"], "労働分配率_終点": ls.loc[b_, "全規模"],
        "始点の分配率を維持した場合の追加人件費_兆円": (ls.loc[a_, "全規模"] * va_all.loc[b_, "付加価値"] - va_all.loc[b_, "人件費"]) / MIL,
    }
pd.DataFrame(dist).T.to_csv(D / "value_added_distribution.csv", encoding="utf-8-sig")

pp = pd.DataFrame({
    "一人当たり付加価値": va_all["付加価値"] / va_all["人数"],
    "一人当たり人件費": va_all["人件費"] / va_all["人数"],
}) * 1e6 / 1e4  # 万円
pp["人数_万人"] = va_all["人数"] / 1e4
pp.index.name = "年度"
pp.round(2).to_csv(D / "per_person_va_labor_cost.csv", encoding="utf-8-sig")
before = ls.loc[1980:2024, "全規模"]
metrics["labor_share"] = {
    "全規模_2012": ls.loc[2012, "全規模"], "全規模_2025": ls.loc[2025, "全規模"],
    "全規模_1980から2024の最小": before.min(), "最小の年度": int(before.idxmin()),
    "全規模_1980から2024の平均": before.mean(), "全規模_2000から2019の平均": ls.loc[2000:2019, "全規模"].mean(),
    "10億円以上_2012": ls.loc[2012, "資本金10億円以上"], "10億円以上_2025": ls.loc[2025, "資本金10億円以上"],
    "10億円未満_2012": ls.loc[2012, "資本金10億円未満"], "10億円未満_2025": ls.loc[2025, "資本金10億円未満"],
    "付加価値に占める10億円以上のシェア_2012": va_big.loc[2012, "付加価値"] / va_all.loc[2012, "付加価値"],
    "付加価値に占める10億円以上のシェア_2025": va_big.loc[2025, "付加価値"] / va_all.loc[2025, "付加価値"],
    "分配": dist,
    "一人当たり付加価値_2012_万円": pp.loc[2012, "一人当たり付加価値"], "一人当たり付加価値_2025_万円": pp.loc[2025, "一人当たり付加価値"],
    "一人当たり人件費_2012_万円": pp.loc[2012, "一人当たり人件費"], "一人当たり人件費_2025_万円": pp.loc[2025, "一人当たり人件費"],
    "一人当たり付加価値_増加率": pp.loc[2025, "一人当たり付加価値"] / pp.loc[2012, "一人当たり付加価値"] - 1,
    "一人当たり人件費_増加率": pp.loc[2025, "一人当たり人件費"] / pp.loc[2012, "一人当たり人件費"] - 1,
    "人数_2012_万人": pp.loc[2012, "人数_万人"], "人数_2025_万人": pp.loc[2025, "人数_万人"],
}
# 構成比の変化の寄与（シェア変化を、規模内の変化と規模間の構成変化に分ける）
w0, w1 = metrics["labor_share"]["付加価値に占める10億円以上のシェア_2012"], metrics["labor_share"]["付加価値に占める10億円以上のシェア_2025"]
s_big0, s_big1 = ls.loc[2012, "資本金10億円以上"], ls.loc[2025, "資本金10億円以上"]
s_sm0, s_sm1 = ls.loc[2012, "資本金10億円未満"], ls.loc[2025, "資本金10億円未満"]
within = (w0 * (s_big1 - s_big0) + (1 - w0) * (s_sm1 - s_sm0))
mix = ((w1 - w0) * (s_big1 - s_sm1))
metrics["labor_share"]["全規模の変化_規模内の変化による_pt"] = within * 100
metrics["labor_share"]["全規模の変化_規模間の構成変化による_pt"] = mix * 100
metrics["labor_share"]["全規模の変化_合計_pt"] = (ls.loc[2025, "全規模"] - ls.loc[2012, "全規模"]) * 100


# ---- 8. 売上高を物価で割る（実質需要の目安） ----
defl = pd.read_csv(D / "deflators_fy.csv", index_col=0)
sales = h["売上高"]
rs = pd.DataFrame({"名目売上高": sales, **{k: defl[k] for k in defl.columns}}).loc[BASE:LAST].dropna()
rs["両指数の平均"] = rs[list(defl.columns)].mean(axis=1)
real = {
    "名目売上高": rs["名目売上高"],
    "実質売上高_企業物価で割る": rs["名目売上高"] / rs["国内企業物価指数_総平均"],
    "実質売上高_サービス価格で割る": rs["名目売上高"] / rs["企業向けサービス価格指数_総平均"],
    "実質売上高_両指数の平均で割る": rs["名目売上高"] / rs["両指数の平均"],
    "国内企業物価指数": rs["国内企業物価指数_総平均"],
    "企業向けサービス価格指数": rs["企業向けサービス価格指数_総平均"],
}
ri = pd.DataFrame(real)
ri = ri / ri.loc[BASE] * 100
ri.index.name = "年度"
ri.round(3).to_csv(D / "real_sales_index.csv", encoding="utf-8-sig")
metrics["real_sales"] = {
    "名目売上高_増加率": ri.loc[LAST, "名目売上高"] / 100 - 1,
    "国内企業物価指数_上昇率": ri.loc[LAST, "国内企業物価指数"] / 100 - 1,
    "企業向けサービス価格指数_上昇率": ri.loc[LAST, "企業向けサービス価格指数"] / 100 - 1,
    "実質売上高_企業物価で割る_増加率": ri.loc[LAST, "実質売上高_企業物価で割る"] / 100 - 1,
    "実質売上高_サービス価格で割る_増加率": ri.loc[LAST, "実質売上高_サービス価格で割る"] / 100 - 1,
    "実質売上高_両指数の平均で割る_増加率": ri.loc[LAST, "実質売上高_両指数の平均で割る"] / 100 - 1,
    "注意": "売上高は取引段階の重複（卸売など）と輸出売上を含む総額。企業物価は国内向けの財、サービス価格は企業向けで、法人企業統計の売上の物価を正確に表さない。実質需要の目安にとどまる。",
}

# ---- 9. 営業利益率の上昇の中身（売上原価率・販管費率） ----
cs = pd.read_csv(RAW / "houjin_table1_all_ex_fin_cost_structure_FY1980_2025.csv").set_index("年度")
assert (cs["売上高"] - cs["売上原価"] - cs["販売費及び一般管理費"] - cs["営業利益"]).abs().max() == 0
ratio = pd.DataFrame({
    "売上原価率": cs["売上原価"] / cs["売上高"],
    "販管費率": cs["販売費及び一般管理費"] / cs["売上高"],
    "営業利益率": cs["営業利益"] / cs["売上高"],
    "人件費率": va_all["人件費"] / cs["売上高"],
    "減価償却費率": va_all["減価償却費計"] / cs["売上高"],
}) * 100
ratio.index.name = "年度"
ratio.round(4).to_csv(D / "cost_ratios.csv", encoding="utf-8-sig")
dr = ratio.loc[LAST] - ratio.loc[BASE]
metrics["cost_structure"] = {
    "売上原価率_2012": ratio.loc[BASE, "売上原価率"], "売上原価率_2025": ratio.loc[LAST, "売上原価率"],
    "販管費率_2012": ratio.loc[BASE, "販管費率"], "販管費率_2025": ratio.loc[LAST, "販管費率"],
    "営業利益率_2012": ratio.loc[BASE, "営業利益率"], "営業利益率_2025": ratio.loc[LAST, "営業利益率"],
    "営業利益率の変化_pt": dr["営業利益率"], "売上原価率の変化_pt": dr["売上原価率"], "販管費率の変化_pt": dr["販管費率"],
    "人件費率の変化_pt": dr["人件費率"], "人件費率_2012": ratio.loc[BASE, "人件費率"], "人件費率_2025": ratio.loc[LAST, "人件費率"],
    "人件費以外の売上原価・販管費の率の変化_pt": (dr["売上原価率"] + dr["販管費率"]) - dr["人件費率"],
    "注意": "人件費は売上原価と販管費の両方に含まれる。売上原価率の低下の何割が人件費由来かは、この表からは分けられない",
}

def clean(o):
    if isinstance(o, dict):
        return {str(k): clean(v) for k, v in o.items()}
    if hasattr(o, "item"):
        return o.item()
    return o


(D / "metrics.json").write_text(json.dumps(clean(metrics), ensure_ascii=False, indent=1), encoding="utf-8")
print(json.dumps(clean({k: v for k, v in metrics.items() if k != "営業利益率_系列"}), ensure_ascii=False, indent=1))

In [ ]:
%%writefile check_article_numbers.py
"""記事に書いた主な数値を、計算結果（data/metrics.json）と突き合わせる。

先に build_data.py と analyze.py を実行しておく。
統計が改定されていると、記事の数値と一致しないことがあります（誤りとは限りません）。

実行: python check_article_numbers.py
"""
import json
import sys
from pathlib import Path

HERE = Path(__file__).resolve().parent
m = json.load(open(HERE / "data" / "metrics.json", encoding="utf-8"))

def check(label, actual, expected, tol):
    ok = abs(actual - expected) <= tol
    print(("OK  " if ok else "NG  ") + f"{label}: 計算値 {actual:.4g}（記事の値 {expected}）")
    return ok

d = m["decomposition"]["2012->2025"]
o = m["operating_profit_split"]
r = m["real_sales"]
c = m["cost_structure"]
n = m["nonop_vs_bop"]
f = m["fx_counterfactual"]
L = m["labor_share"]
D = L["分配"]["2012->2025"]
results = [
    check("経常利益 2012年度（兆円）", d["経常利益_始点_兆円"], 48.5, 0.05),
    check("経常利益 2025年度（兆円）", d["経常利益_終点_兆円"], 124.8, 0.05),
    check("増加分に占める営業利益の割合", d["営業利益_割合"], 0.70, 0.005),
    check("増加分に占める営業外収益の割合", d["営業外収益_割合"], 0.41, 0.005),
    check("増加分に占める営業外費用の割合", d["営業外費用_割合"], -0.11, 0.005),
    check("売上高の名目増加率", o["売上高_増加率"], 0.25, 0.005),
    check("国内企業物価指数の上昇率", r["国内企業物価指数_上昇率"], 0.31, 0.005),
    check("企業向けサービス価格指数の上昇率", r["企業向けサービス価格指数_上昇率"], 0.21, 0.005),
    check("実質売上高の増加率（両指数の平均で割る）", r["実質売上高_両指数の平均で割る_増加率"], -0.007, 0.0005),
    check("営業利益率 2012年度", o["営業利益率_始点"], 0.0291, 0.00005),
    check("営業利益率 2025年度", o["営業利益率_終点"], 0.0543, 0.00005),
    check("売上増の効果・平均（兆円）", o["売上増の効果_平均_兆円"], 14.4, 0.05),
    check("利益率上昇の効果・平均（兆円）", o["利益率上昇の効果_平均_兆円"], 39.0, 0.05),
    check("利益率上昇の割合・平均", o["利益率上昇の割合_平均"], 0.73, 0.005),
    check("利益率上昇の割合・下限（売上を2012年度の水準で評価）", o["利益率上昇の割合_下限"], 0.65, 0.005),
    check("利益率上昇の割合・上限", o["利益率上昇の割合_上限"], 0.81, 0.005),
    check("売上原価率の変化（ポイント）", c["売上原価率の変化_pt"], -4.14, 0.005),
    check("販管費率の変化（ポイント）", c["販管費率の変化_pt"], 1.62, 0.005),
    check("人件費率の変化（ポイント）", c["人件費率の変化_pt"], -0.65, 0.005),
    check("配当金等の増分／営業外収益の増分", n["配当金配分済支店収益の増分／営業外収益の増分"], 0.46, 0.005),
    check("証券投資収益（受取）の増分（兆円）", n["証券投資収益_増分_兆円"], 15.4, 0.05),
    check("直接投資の配当金・利子と証券投資収益の増分合計（兆円）", n["直接投資の配当金・利子と証券投資収益の増分合計_兆円"], 30.6, 0.05),
    check("為替の換算差・全額ドル建ての試算（兆円）", f["最終年の円安の寄与_兆円"], 14.3, 0.05),
    check("うち再投資収益の分（兆円）", f["換算差の内訳_再投資収益_兆円"], 5.8, 0.05),
    check("2012年度の為替での2025年度の直接投資収益（兆円）", f["反実仮想_2012年度の為替での2025年度の直接投資収益_兆円"], 17.5, 0.05),
    check("労働分配率 2012年度", L["全規模_2012"], 0.723, 0.0005),
    check("労働分配率 2025年度", L["全規模_2025"], 0.630, 0.0005),
    check("労働分配率 10億円以上 2012年度", L["10億円以上_2012"], 0.605, 0.0005),
    check("労働分配率 10億円以上 2025年度", L["10億円以上_2025"], 0.465, 0.0005),
    check("労働分配率 10億円未満 2012年度", L["10億円未満_2012"], 0.775, 0.0005),
    check("労働分配率 10億円未満 2025年度", L["10億円未満_2025"], 0.713, 0.0005),
    check("付加価値の増加（兆円）", D["付加価値_増分_兆円"], 101.3, 0.05),
    check("増加分に占める営業純益の割合", D["営業純益_割合"], 0.49, 0.005),
    check("増加分に占める人件費の割合", D["人件費_割合"], 0.38, 0.005),
    check("一人当たり付加価値の増加率", L["一人当たり付加価値_増加率"], 0.29, 0.005),
    check("一人当たり人件費の増加率", L["一人当たり人件費_増加率"], 0.12, 0.005),
    check("2012年度の分配率を保った場合の追加人件費（兆円）", D["始点の分配率を維持した場合の追加人件費_兆円"], 34.9, 0.05),
]
print()
print("すべて一致" if all(results) else "不一致があります。上の NG を確認してください")
sys.exit(0 if all(results) else 1)

In [ ]:
%%writefile charts/make_charts.py
"""記事の全図を作る。data/ のCSV（build_data.py, analyze.py の出力）だけを読む。

実行: python charts/make_charts.py
"""
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib import font_manager

ROOT = Path(__file__).resolve().parents[1]
D = ROOT / "data"
OUT = ROOT / "charts"
MIL = 1e6

BLUE, ORANGE, GRAY, DARK, TEAL, RED = "#2875b9", "#d56b2d", "#656565", "#222222", "#1f8a70", "#b34d79"
SRC_HOUJIN = "出所：財務省「法人企業統計調査」（金融業・保険業を除く全産業・全規模、年度）"


def choose_font():
    names = {f.name for f in font_manager.fontManager.ttflist}
    for name in ["Yu Gothic", "Noto Sans CJK JP", "Noto Sans JP", "IPAexGothic", "IPAGothic"]:
        if name in names:
            plt.rcParams["font.family"] = name
            break
    else:
        import japanize_matplotlib  # noqa: F401
    plt.rcParams.update({"axes.unicode_minus": False, "font.size": 13, "text.color": "#222222"})


def frame(title, subtitle, note, source, ncols=1, figsize=(11, 7.0)):
    fig, axes = plt.subplots(1, ncols, figsize=figsize, dpi=190, squeeze=False)
    fig.patch.set_facecolor("white")
    for ax in axes[0]:
        for s in ["top", "right"]:
            ax.spines[s].set_visible(False)
        for s in ["bottom", "left"]:
            ax.spines[s].set_color("#aaaaaa")
        ax.grid(axis="y", color="#e2e2e2", lw=0.7, zorder=0)
        ax.set_axisbelow(True)
    fig.text(0.07, 0.96, title, ha="left", va="top", fontsize=20, weight="bold")
    fig.text(0.07, 0.902, subtitle, ha="left", va="top", fontsize=13.5, color="#555555")
    fig.text(0.07, 0.035, note + "\n" + source, ha="left", va="bottom", fontsize=13, color="#4a4a4a", linespacing=1.55)
    fig.subplots_adjust(left=0.10, right=0.965, top=0.80, bottom=0.27, wspace=0.18)
    return fig, (axes[0] if ncols > 1 else axes[0][0])


def save(fig, name):
    OUT.mkdir(exist_ok=True)
    fig.savefig(OUT / name, facecolor="white")
    plt.close(fig)


def end_label(ax, x, y, text, color, dy=0, dx=8, size=14.5):
    ax.annotate(text, (x, y), xytext=(dx, dy), textcoords="offset points", color=color, fontsize=size, va="center")


def main():
    choose_font()
    m = pd.read_csv(D / "merged_fy_million_yen_and_oku_yen.csv", index_col=0)
    h = pd.read_csv(ROOT / "data/raw/houjin_table1_all_ex_fin_FY1980_2025.csv").set_index("年度")
    ke, op, noi, noe = (h[c] / MIL for c in ["経常利益", "営業利益", "営業外収益", "営業外費用"])
    ie = h["支払利息等"] / MIL

    # 図1 経常利益と営業利益
    fig, ax = frame("図1　経常利益は過去最高を更新している", "1980〜2025年度・兆円（名目）",
                    "注：全産業（金融業・保険業を除く）。経常利益＝営業利益＋営業外収益−営業外費用。\n名目値で、物価の影響は調整していない。", SRC_HOUJIN)
    ax.plot(ke.index, ke, color=DARK, lw=2.6)
    ax.plot(op.index, op, color=BLUE, lw=2.2, ls="--")
    ax.plot([2012], [ke[2012]], "o", color=DARK, ms=6)
    ax.annotate("2012年度\n48.5兆円", (2012, ke[2012]), xytext=(-8, 28), textcoords="offset points", ha="right", fontsize=13.5, arrowprops=dict(arrowstyle="-", color="#999999"))
    end_label(ax, 2025, ke[2025], f"経常利益\n{ke[2025]:.1f}兆円", DARK, dy=8)
    end_label(ax, 2025, op[2025], f"営業利益\n{op[2025]:.1f}兆円", BLUE, dy=-12)
    ax.set_xlim(1980, 2038)
    ax.set_xticks([1980, 1990, 2000, 2010, 2020, 2025])
    ax.set_ylabel("兆円")
    ax.set_ylim(0, 135)
    save(fig, "fig1_ordinary_profit.png")

    # 図2 ウォーターフォール
    dd = pd.read_csv(D / "decomposition.csv", index_col=0).loc["2012->2025"]
    a, b = dd["経常利益_始点_兆円"], dd["経常利益_終点_兆円"]
    d1, d2, d3 = dd["営業利益_兆円"], dd["営業外収益_兆円"], dd["営業外費用の増加_兆円"]
    fig, ax = frame("図2　最高益の増加分は、営業利益が約7割、営業外収益が約4割", "経常利益の増加分（2012年度→2025年度）の内訳・兆円",
                    "注：全産業（金融業・保険業を除く）、名目値。営業外費用の増加は差し引くので、\n割合は営業利益＋営業外収益−営業外費用＝100％になる。", SRC_HOUJIN + "より計算")
    xs = range(5)
    labels = ["2012年度\n経常利益", "営業利益の\n増加", "営業外収益の\n増加", "営業外費用の\n増加", "2025年度\n経常利益"]
    bottoms = [0, a, a + d1, a + d1 + d2 - d3, 0]
    heights = [a, d1, d2, d3, b]
    colors = [DARK, BLUE, ORANGE, GRAY, DARK]
    ax.bar(xs, heights, bottom=bottoms, color=colors, width=0.58, zorder=3)
    tops = [a, a + d1, a + d1 + d2, a + d1 + d2, b]
    texts = [f"{a:.1f}", f"+{d1:.1f}\n（{dd['営業利益_割合']*100:.0f}％）", f"+{d2:.1f}\n（{dd['営業外収益_割合']*100:.0f}％）",
             f"−{d3:.1f}\n（−{-dd['営業外費用_割合']*100:.0f}％）", f"{b:.1f}"]
    for x, t, y in zip(xs, texts, tops):
        ax.text(x, y + 2.5, t, ha="center", va="bottom", fontsize=15, weight="bold", color=DARK)
    ax.set_xticks(list(xs))
    ax.set_xticklabels(labels, fontsize=13.5)
    ax.set_ylabel("兆円")
    ax.set_ylim(0, 150)
    save(fig, "fig2_waterfall.png")

    # 図3 営業外収益と国際収支の直接投資収益
    bp = m[["直接投資収益_受取", "直接投資_配当金配分済支店収益_受取", "直接投資_再投資収益_受取"]].dropna() / 1e4
    fig, ax = frame("図3　営業外収益は31兆円増、海外からの配当等は全部門で15兆円増", "営業外収益（法人企業統計）と直接投資収益（国際収支）・兆円",
                    "注：直接投資収益は全部門の受取で、金融機関も含むため、法人企業統計の対象より広い。\n再投資収益は内部留保の推計で、法人企業統計の営業外収益には現れない。\n2024・2025年度の再投資収益は暫定（点線）。外国証券からの配当・利子は含めていない。",
                    "出所：財務省「法人企業統計調査」、日本銀行「国際収支統計」（BPM6）")
    x0 = 1996
    ax.plot(noi.loc[x0:].index, noi.loc[x0:], color=ORANGE, lw=2.8)
    ax.plot(bp.loc[x0:].index, bp.loc[x0:, "直接投資収益_受取"], color=BLUE, lw=2.2)
    ax.plot(bp.loc[x0:].index, bp.loc[x0:, "直接投資_配当金配分済支店収益_受取"], color="#164e80", lw=2.0, ls="--")
    r = bp.loc[x0:, "直接投資_再投資収益_受取"]
    ax.plot(r.loc[:2023].index, r.loc[:2023], color=TEAL, lw=2.0, ls=":")
    ax.plot(r.loc[2023:].index, r.loc[2023:], color=TEAL, lw=1.4, ls=(0, (1, 3)), alpha=0.7)
    end_label(ax, 2025, noi[2025], f"営業外収益\n{noi[2025]:.1f}", ORANGE, dy=6, size=14)
    end_label(ax, 2025, bp.loc[2025, "直接投資収益_受取"], f"直接投資収益（受取）\n{bp.loc[2025,'直接投資収益_受取']:.1f}", BLUE, dy=2, size=14)
    end_label(ax, 2025, bp.loc[2025, "直接投資_配当金配分済支店収益_受取"], f"うち配当金など\n{bp.loc[2025,'直接投資_配当金配分済支店収益_受取']:.1f}", "#164e80", dy=10, size=14)
    end_label(ax, 2025, bp.loc[2025, "直接投資_再投資収益_受取"], f"うち再投資収益\n{bp.loc[2025,'直接投資_再投資収益_受取']:.1f}", TEAL, dy=-34, size=14)
    ax.set_xlim(1996, 2040)
    ax.set_xticks([1996, 2000, 2005, 2010, 2015, 2020, 2025])
    ax.set_ylabel("兆円")
    ax.set_ylim(0, 60)
    save(fig, "fig3_nonop_vs_bop.png")

    # 図4 円安の寄与
    cf = pd.read_csv(D / "fx_counterfactual.csv", index_col=0)
    fig, ax = frame("図4　為替を2012年度に固定すると、直接投資収益の円換算は14兆円小さい", "直接投資収益（受取）の円換算：実際と、2012年度の為替レートに固定した場合・兆円",
                    "注：直接投資収益の全額がドル建てと仮定した試算。再投資収益の分（約6兆円）は単体の営業外収益に現れない。\n営業利益に含まれる輸出採算への円安効果や、ドル以外の通貨の動きは含まない。\nドル円は年度平均（2012年度83円、2025年度151円）。",
                    "出所：日本銀行「国際収支統計」「時系列統計データ検索サイト」より計算")
    ax.fill_between(cf.index, cf["円レート固定_兆円"], cf["直接投資収益_受取_兆円"], color=ORANGE, alpha=0.22, zorder=2)
    ax.plot(cf.index, cf["直接投資収益_受取_兆円"], color=BLUE, lw=2.6)
    ax.plot(cf.index, cf["円レート固定_兆円"], color=GRAY, lw=2.2, ls="--")
    last = cf.index[-1]
    end_label(ax, last, cf.loc[last, "直接投資収益_受取_兆円"], f"実際\n{cf.loc[last,'直接投資収益_受取_兆円']:.1f}", BLUE, dy=4)
    end_label(ax, last, cf.loc[last, "円レート固定_兆円"], f"2012年度の\nレート固定\n{cf.loc[last,'円レート固定_兆円']:.1f}", GRAY, dy=-6)
    ax.text(2023.2, 22.0, f"為替の換算差\n約{cf.loc[last,'円安の寄与_兆円']:.0f}兆円", ha="center", va="center", color="#9a4a1c", fontsize=15, weight="bold", zorder=5)
    ax.set_xlim(2012, 2029)
    ax.set_xticks([2012, 2015, 2018, 2021, 2024, 2025])
    ax.set_ylabel("兆円")
    ax.set_ylim(0, 38)
    save(fig, "fig4_fx_counterfactual.png")

    # 図5 SNAとの突き合わせ
    c = pd.read_csv(D / "sna_vs_houjin.csv", index_col=0)
    fig, axes = frame("図5　SNAの営業余剰（純）は横ばい、法人企業統計の営業利益は2.6倍", "非金融法人企業の営業利益・営業余剰（1994〜2024年・兆円）",
                      "注：法人企業統計は年度、SNAは暦年で、年度tを暦年tに対応させた。範囲も異なる\n（法人企業統計は金融業・保険業を除く、SNAは非金融法人企業）。\nSNAの固定資本減耗は時価、法人企業統計の減価償却費は帳簿価額。",
                      "出所：財務省「法人企業統計調査」、内閣府「国民経済計算」2024年度年次推計（2020年基準）", ncols=2)
    ax1, ax2 = axes
    for ax in axes:
        ax.set_xlim(1994, 2024)
        ax.set_xticks([1994, 2004, 2014, 2024])
        ax.set_ylim(0, 185)
    ax1.set_ylabel("兆円")
    ax1.set_title("総（減価償却・固定資本減耗の控除前）", fontsize=14, loc="left", color="#444444")
    ax1.plot(c.index, c["法人企業統計_営業利益＋減価償却費_兆円"], color=BLUE, lw=2.4)
    ax1.plot(c.index, c["SNA_営業余剰_総_兆円"], color=DARK, lw=2.4, ls="--")
    ax1.text(1995, 156, "SNA 営業余剰（総）", color=DARK, fontsize=13.5)
    ax1.text(1995, 40, "法人企業統計\n営業利益＋減価償却費", color=BLUE, fontsize=13.5, va="center")
    ax2.set_title("純（減価償却・固定資本減耗の控除後）", fontsize=14, loc="left", color="#444444")
    ax2.plot(c.index, c["法人企業統計_営業利益_兆円"], color=BLUE, lw=2.4)
    ax2.plot(c.index, c["SNA_営業余剰_純_兆円"], color=DARK, lw=2.4, ls="--")
    ax2.text(1995, 90, "SNA 営業余剰（純）", color=DARK, fontsize=13.5)
    ax2.text(1995, 12, "法人企業統計 営業利益", color=BLUE, fontsize=13.5, va="center")
    save(fig, "fig5_sna_vs_houjin.png")

    # 図6 支払利息等
    fig, ax = frame("図6　支払利息は営業利益の6割から1割へ減り、足元でやや戻した", "営業利益と支払利息等（1980〜2025年度・兆円）",
                    "注：全産業（金融業・保険業を除く）。支払利息等は営業外費用の主な内訳（割引料を含む）。\n営業外費用のうち支払利息等以外の内訳は、この表では分からない。", SRC_HOUJIN)
    ax.plot(op.index, op, color=BLUE, lw=2.4)
    ax.plot(ie.index, ie, color=GRAY, lw=2.4)
    for y in (1995, 2012, 2025):
        ratio = ie[y] / op[y] * 100
        ax.plot([y], [ie[y]], "o", color=GRAY, ms=5)
        ax.annotate(f"{y}年度\n営業利益の{ratio:.0f}％", (y, ie[y]), xytext=(0, -12), textcoords="offset points", ha="center", va="top", fontsize=13, color="#444444")
    end_label(ax, 2025, op[2025], f"営業利益\n{op[2025]:.1f}", BLUE, dy=4)
    end_label(ax, 2025, ie[2025], f"支払利息等\n{ie[2025]:.1f}", GRAY, dy=16)
    ax.set_xlim(1980, 2036)
    ax.set_xticks([1980, 1990, 2000, 2010, 2020, 2025])
    ax.set_ylabel("兆円")
    ax.set_ylim(-22, 105)
    save(fig, "fig6_interest.png")
    # 図7 労働分配率
    ls = pd.read_csv(D / "labor_share_by_size.csv", index_col=0) * 100
    fig, ax = frame("図7　労働分配率は63％まで下がり、1980年度以降で最低", "労働分配率＝人件費÷付加価値（1980〜2025年度・％）",
                    "注：人件費は役員・従業員の給与・賞与と福利厚生費。付加価値＝人件費＋支払利息等＋賃借料\n＋租税公課＋営業純益。金融業・保険業を除く全産業。資本金10億円未満は全規模から10億円以上を引いて算出。\n役員賞与はFY2007から人件費に含まれる（表の区分の変更で、影響は付加価値の0.2〜0.4％程度）。", SRC_HOUJIN + "より計算")
    ax.plot(ls.index, ls["資本金10億円未満"], color=GRAY, lw=2.0, ls="-.")
    ax.plot(ls.index, ls["資本金10億円以上"], color=ORANGE, lw=2.0, ls="--")
    ax.plot(ls.index, ls["全規模"], color=DARK, lw=2.8)
    ax.plot([2012], [ls.loc[2012, "全規模"]], "o", color=DARK, ms=6)
    ax.annotate(f"2012年度 {ls.loc[2012,'全規模']:.1f}％", (2012, ls.loc[2012, "全規模"]), xytext=(2014.5, 63.5), textcoords="data", ha="left", va="center", fontsize=13.5,
                arrowprops=dict(arrowstyle="-", color="#999999"))
    end_label(ax, 2025, ls.loc[2025, "資本金10億円未満"], f"資本金10億円未満\n{ls.loc[2025,'資本金10億円未満']:.1f}％", GRAY, dy=6, size=13.5)
    end_label(ax, 2025, ls.loc[2025, "全規模"], f"全規模\n{ls.loc[2025,'全規模']:.1f}％", DARK, dy=-2, size=14.5)
    end_label(ax, 2025, ls.loc[2025, "資本金10億円以上"], f"資本金10億円以上\n{ls.loc[2025,'資本金10億円以上']:.1f}％", ORANGE, dy=0, size=13.5)
    ax.set_xlim(1980, 2043)
    ax.set_xticks([1980, 1990, 2000, 2010, 2020, 2025])
    ax.set_ylabel("％")
    ax.set_ylim(40, 85)
    save(fig, "fig7_labor_share.png")

    # 図8 付加価値の増加分の行き先
    vd = pd.read_csv(D / "value_added_distribution.csv", index_col=0).loc["2012->2025"]
    items = [("営業純益", "営業純益_兆円", "営業純益_割合", ORANGE), ("人件費", "人件費_兆円", "人件費_割合", BLUE),
             ("動産・不動産\n賃借料", "動産不動産賃借料_兆円", "動産不動産賃借料_割合", GRAY), ("租税公課", "租税公課_兆円", "租税公課_割合", GRAY),
             ("支払利息等", "支払利息等_兆円", "支払利息等_割合", GRAY)]
    fig, ax = frame("図8　付加価値の増加分の約5割は営業純益、人件費は約4割", f"付加価値の増加分（2012年度→2025年度、{vd['付加価値_増分_兆円']:.1f}兆円）の行き先・兆円",
                    "注：付加価値＝人件費＋支払利息等＋賃借料＋租税公課＋営業純益。営業純益＝営業利益−支払利息等。\n金融業・保険業を除く全産業、名目値。割合の合計は100％。", SRC_HOUJIN + "より計算")
    for i, (lab, kv, kr, col) in enumerate(items):
        ax.bar(i, vd[kv], color=col, width=0.58, zorder=3)
        ax.text(i, vd[kv] + 1.0, f"+{vd[kv]:.1f}\n（{vd[kr]*100:.0f}％）", ha="center", va="bottom", fontsize=15, weight="bold", color=DARK)
    ax.set_xticks(range(len(items)))
    ax.set_xticklabels([x[0] for x in items], fontsize=13.5)
    ax.set_ylabel("兆円")
    ax.set_ylim(0, 62)
    save(fig, "fig8_value_added_distribution.png")

    # 図9 一人当たり付加価値と人件費
    pp = pd.read_csv(D / "per_person_va_labor_cost.csv", index_col=0).loc[2012:]
    idx = pp[["一人当たり付加価値", "一人当たり人件費"]] / pp[["一人当たり付加価値", "一人当たり人件費"]].loc[2012] * 100
    fig, ax = frame("図9　一人当たり付加価値は29％増、一人当たり人件費は12％増", "従業員・役員一人当たり（2012年度＝100）",
                    f"注：名目値で、物価の影響は調整していない。人数は期中平均の従業員数と役員数の合計。2012年度→2025年度で、\n一人当たり付加価値は{pp.loc[2012,'一人当たり付加価値']:.0f}万円→{pp.loc[2025,'一人当たり付加価値']:.0f}万円、"
                    f"一人当たり人件費は{pp.loc[2012,'一人当たり人件費']:.0f}万円→{pp.loc[2025,'一人当たり人件費']:.0f}万円。金融業・保険業を除く全産業。", SRC_HOUJIN + "より計算")
    ax.axhline(100, color="#999999", ls=":", lw=0.9)
    ax.plot(idx.index, idx["一人当たり付加価値"], color=ORANGE, lw=2.6)
    ax.plot(idx.index, idx["一人当たり人件費"], color=BLUE, lw=2.6, ls="--")
    end_label(ax, 2025, idx.iloc[-1]["一人当たり付加価値"], f"一人当たり付加価値\n{idx.iloc[-1]['一人当たり付加価値']:.1f}", ORANGE, dy=2, size=14)
    end_label(ax, 2025, idx.iloc[-1]["一人当たり人件費"], f"一人当たり人件費\n{idx.iloc[-1]['一人当たり人件費']:.1f}", BLUE, dy=-2, size=14)
    ax.set_xlim(2012, 2030)
    ax.set_xticks([2012, 2015, 2018, 2021, 2024, 2025])
    ax.set_ylabel("2012年度＝100")
    ax.set_ylim(90, 135)
    save(fig, "fig9_per_person.png")
    # 図10 名目売上高と、物価で割った売上高
    ri = pd.read_csv(D / "real_sales_index.csv", index_col=0)
    fig, ax = frame("図10　売上高は名目で25％増だが、物価で割ると横ばい", "売上高と物価の推移（2012年度＝100）",
                    "注：売上高は取引段階の重複や輸出売上を含む総額。実質売上高は、国内企業物価指数（財）と企業向けサービス\n価格指数の平均で割った目安で、法人企業統計の売上の物価を正確に表すものではない。金融業・保険業を除く全産業。",
                    "出所：財務省「法人企業統計調査」、日本銀行「企業物価指数」「企業向けサービス価格指数」より計算")
    ax.axhline(100, color="#999999", ls=":", lw=0.9)
    ax.plot(ri.index, ri["名目売上高"], color=DARK, lw=2.8)
    ax.plot(ri.index, ri["実質売上高_両指数の平均で割る"], color=ORANGE, lw=2.8)
    ax.plot(ri.index, ri["国内企業物価指数"], color=GRAY, lw=1.8, ls="--")
    ax.plot(ri.index, ri["企業向けサービス価格指数"], color=GRAY, lw=1.8, ls=":")
    last = ri.index[-1]
    end_label(ax, last, ri.loc[last, "国内企業物価指数"], f"企業物価（財） {ri.loc[last,'国内企業物価指数']:.1f}", GRAY, dy=9, size=13.5)
    end_label(ax, last, ri.loc[last, "名目売上高"], f"名目売上高 {ri.loc[last,'名目売上高']:.1f}", DARK, dy=0, size=14.5)
    end_label(ax, last, ri.loc[last, "企業向けサービス価格指数"], f"サービス価格 {ri.loc[last,'企業向けサービス価格指数']:.1f}", GRAY, dy=-10, size=13.5)
    end_label(ax, last, ri.loc[last, "実質売上高_両指数の平均で割る"], f"実質売上高（目安） {ri.loc[last,'実質売上高_両指数の平均で割る']:.1f}", ORANGE, dy=0, size=14.5)
    ax.set_xlim(2012, 2033)
    ax.set_xticks([2012, 2015, 2018, 2021, 2024, 2025])
    ax.set_ylabel("2012年度＝100")
    ax.set_ylim(90, 138)
    save(fig, "fig10_real_sales.png")
    # 図11 営業利益率の上昇の中身
    cr = pd.read_csv(D / "cost_ratios.csv", index_col=0)
    r0, r1 = cr.loc[2012], cr.loc[2025]
    lab = r0["人件費率"] - r1["人件費率"]
    oth = (r0["売上原価率"] + r0["販管費率"] - r0["人件費率"]) - (r1["売上原価率"] + r1["販管費率"] - r1["人件費率"])
    fig, ax = frame("図11　利益率の上昇2.5ポイントのうち、人件費の寄与は約4分の1", "営業利益率の変化（2012年度→2025年度・売上高に対する％）",
                    "注：営業利益率＝営業利益÷売上高。人件費は売上原価と販管費の両方に含まれるため、費用全体の率を\n人件費の率と、それ以外（仕入・材料・経費など）の率に分けた。金融業・保険業を除く全産業。",
                    SRC_HOUJIN + "より計算")
    xs = range(4)
    heights = [r0["営業利益率"], lab, oth, r1["営業利益率"]]
    bottoms = [0, r0["営業利益率"], r0["営業利益率"] + lab, 0]
    colors = [DARK, BLUE, ORANGE, DARK]
    ax.bar(xs, heights, bottom=bottoms, color=colors, width=0.58, zorder=3)
    tops = [heights[0], bottoms[1] + lab, bottoms[2] + oth, heights[3]]
    texts = [f"{r0['営業利益率']:.1f}％", f"+{lab:.2f}\n（{lab/(lab+oth)*100:.0f}％）", f"+{oth:.2f}\n（{oth/(lab+oth)*100:.0f}％）", f"{r1['営業利益率']:.1f}％"]
    for x, t_, y in zip(xs, texts, tops):
        ax.text(x, y + 0.12, t_, ha="center", va="bottom", fontsize=15, weight="bold", color=DARK)
    ax.set_xticks(list(xs))
    ax.set_xticklabels(["2012年度\n営業利益率", "人件費率の\n低下", "人件費以外の\n費用率の低下", "2025年度\n営業利益率"], fontsize=13.5)
    ax.set_ylabel("％（ポイント）")
    ax.set_ylim(0, 6.9)
    save(fig, "fig11_margin_sources.png")
    print("saved:", sorted(p.name for p in OUT.glob("*.png")))


if __name__ == "__main__":
    main()

## 1. 公式の入手先から、日銀・内閣府・財務省のデータを取得する

`fetch_data.py` が、次を取得します。取得したデータの指紋を、記事執筆時の値と比べて表示します（不一致は、統計の改定の可能性を示すだけで、誤りとは限りません）。

In [ ]:
import os, subprocess, sys
def run(script):
    env = {**os.environ, "PYTHONIOENCODING": "utf-8"}
    r = subprocess.run([sys.executable, script], capture_output=True, text=True, encoding="utf-8", errors="replace", env=env)
    print(f"--- {script}（終了コード {r.returncode}）")
    print((r.stdout or "").strip())
    if r.returncode != 0:
        print(r.stderr[-1500:])
        raise SystemExit(f"{script} が失敗しました")

run("fetch_data.py")

## 2. 法人企業統計（e-Stat）を用意する

法人企業統計は、次の2つの方法のどちらかで用意します。

**方法A（推奨）：e-Stat API で自動取得する。** e-Stat の [API 機能](https://www.e-stat.go.jp/api/)で、利用登録（無料）をして appId を取得してください。次のセルで appId を聞かれます（Colab では、左の「シークレット」に `ESTAT_APP_ID` という名前で登録しておくと、入力が要りません）。**appId は他人に渡さず、ノートブックやリポジトリに書き込まないでください。**

**方法B：画面からCSVをダウンロードする。** appId がない場合は、次の表のとおりにCSVを3つダウンロードして、アップロードします。同じ表を、業種と規模の選び方を変えて3回ダウンロードします。

- **表：** 法人企業統計調査 時系列データ 表1「金融業、保険業以外の業種（原数値）」。年度（4月〜翌年3月）の表です。
- **期間：** 1980年度から2025年度まで。画面の既定（1960年度から最新年度まで）のまま、全年度で構いません。
- **項目：** 全項目のまま（画面右側の「表示項目選択」は変えません）。

| 画面での選択（「業種」「規模」） | 何に使うか |
|---|---|
| 業種＝全産業（除く金融保険業）、規模＝全規模 | 記事の中心（経常利益の分解、売上、費用率、労働分配率、付加価値） |
| 業種＝純粋持株会社、規模＝全規模 | 純粋持株会社を除いた確認 |
| 業種＝全産業（除く金融保険業）、規模＝10億円以上 | 労働分配率の規模別 |

手順：
1. [e-Stat 法人企業統計調査 時系列データ 表1](https://www.e-stat.go.jp/dbview?sid=0003060791) を開きます。
2. 「業種」「規模」を上の表のとおりに選び、「再表示」を押します。項目と年度は変えません。
3. 画面の「ダウンロード」から、CSVまたはExcel形式を選んで保存します。
4. 方法Aで取得できなかった場合のみ、2つ下のセルで、ファイルを1つずつ選ぶ画面が出ます。上の表の順に、ダウンロードしたファイルを選んでください（CSVでもExcelでも構いません。どの表かは、ファイルの中身から自動で判定します）。

確かめ方（全産業・全規模、2025年度、百万円）：経常利益 124,831,342、売上高 1,720,255,968、付加価値 373,670,064。

In [ ]:
# 方法A：e-Stat API（appId があるとき）。appId は画面に表示されず、ファイルにも保存されません
import os, getpass
from pathlib import Path
import parse_estat
_files = list(Path("data/estat").glob("*.csv")) + list(Path("data/estat").glob("*.xlsx"))
if {parse_estat.classify(f) for f in _files} >= {"all", "holding", "large"}:
    print("e-Statのファイルは、すべてそろっています")
else:
    app_id = os.environ.get("ESTAT_APP_ID")
    if not app_id:
        try:
            from google.colab import userdata
            app_id = userdata.get("ESTAT_APP_ID")
        except Exception:
            app_id = None
    if not app_id:
        app_id = getpass.getpass("e-Stat の appId（無い場合は Enter。次のセルでCSVをアップロードします）: ").strip()
    if app_id:
        os.environ["ESTAT_APP_ID"] = app_id
        run("fetch_estat_api.py")
    else:
        print("appId がないので、次のセルでCSVをアップロードします")

**方法B（CSVのアップロード）：** 上のセルで取得できた場合は、次のセルは何もしません。

In [ ]:
from pathlib import Path
import parse_estat
DESC = parse_estat.DESC
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

def have():
    fs = list(Path("data/estat").glob("*.csv")) + list(Path("data/estat").glob("*.xlsx"))
    return {parse_estat.classify(f) for f in fs} - {None}

for key in ("all", "holding", "large"):
    if key in have():
        print("そろっています:", DESC[key])
        continue
    if IN_COLAB:
        print("ダウンロードしたファイルを選んでください（CSVでもExcelでも可）：", DESC[key])
        up = files.upload()
        if len(up) != 1:
            raise SystemExit("ファイルを1つだけ選んでください。もう一度このセルを実行してください")
        (name, content), = up.items()
        with open(f"data/estat/{name}", "wb") as f:
            f.write(content)
        if parse_estat.classify(f"data/estat/{name}") != key:
            print("注意: このファイルは、期待した表（" + DESC[key] + "）として判定されませんでした。業種・規模の選択を確かめてください")
    else:
        raise SystemExit("data/estat/ に、次の表のファイルを置いてください（CSVかExcel）：" + DESC[key])
print("3つそろいました")

## 3. 分析用データを作り、計算して、図を作る

In [ ]:
run("parse_estat.py")
run("build_data.py")
run("analyze.py")
run("charts/make_charts.py")

## 4. 図を表示する

記事の図1〜図11です。

In [ ]:
import glob, re
from IPython.display import Image, display
files_ = sorted(glob.glob("charts/fig*.png"), key=lambda p: int(re.search(r"fig(\d+)_", p).group(1)))
for p in files_:
    print(p)
    display(Image(p, width=760))

## 5. 記事の主な数値と照合する

記事に書いた数値を、計算結果と突き合わせます。すべて「OK」なら、記事の数値が再現されたことになります。統計が改定されていると、一部が「NG」になることがあります（誤りとは限りません）。

In [ ]:
r = subprocess.run([sys.executable, "check_article_numbers.py"], capture_output=True, text=True, encoding="utf-8", errors="replace", env={**os.environ, "PYTHONIOENCODING": "utf-8"})
print(r.stdout)

## 読み方の注意

- 営業利益の増加分を売上の増加と利益率の上昇に分ける割合は、計算の順序で65%から81%の間で動きます。記事では二通りの平均（利益率73%、売上27%）を使っています。
- 実質売上高は、国内企業物価指数と企業向けサービス価格指数の平均で割った目安です。
- 為替の換算差は、直接投資収益の全額がドル建てと仮定したシナリオ計算です（上限ではありません）。
- 労働分配率の水準は、付加価値の定義で変わります。ここでは法人企業統計の付加価値の内訳（人件費＋支払利息等＋賃借料＋租税公課＋営業純益）に沿っています。